- Consistently found important features and used in regime identity
- top_features_list is static in this notbook and can be generated using Top_Features_Intersection.ipynb
- Relying on LGBM primary model to get top features
- Threshold optimization by profit
- No hyperparameter tuning
- Regime is defined by the top features using GMM
- No feature selection beyong the top_feature_list
- Multiplicative interactions
- Training Period: 2021-01-01 - 2022-12-31 | Validation Period: 2023-01-01 - 2023-12-29.

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
import shap
import catboost as cat
import xgboost as xgb
import lightgbm as lgb
from datetime import timedelta, datetime
import warnings, random, os
warnings.filterwarnings("ignore")

seed = 1
random.seed(seed)
np.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

df = pd.read_csv('SE5.csv')
df['Time'] = pd.to_datetime(df['Time'])
df = df.sort_values(by='Time', ascending=True)
train_val_df = df[df["Time"] < "2024-01-01"]
test_df = df[df["Time"] >= "2024-01-01"]
forward_test_df = pd.read_csv('SE5-2.csv')

X = train_val_df.drop(columns=['class'])
y = train_val_df['class']

print(f"Train/Validation data shape: {train_val_df.shape}")
print(f"Train/Validation data shape: {test_df.shape}")

Train/Validation data shape: (40884, 717)
Train/Validation data shape: (2313, 717)


In [2]:
# # Catboost
# top_features_list = [
#     [
#         "Feature_1",
#         "Feature_2",
#         "Feature_112",
#         "Feature_113",
#         "Feature_176",
#         "Feature_212",
#         "Feature_320",
#         "Feature_347",
#         "Feature_474",
#         "Feature_489",
#         "Feature_715",
#     ],
#     [
#         "Feature_1",
#         "Feature_109",
#         "Feature_237",
#         "Feature_241",
#         "Feature_261",
#         "Feature_453",
#         "Feature_474",
#         "Feature_478",
#         "Feature_710",
#         "Feature_711",
#     ],
#     [
#         "Feature_20",
#         "Feature_235",
#         "Feature_281",
#         "Feature_478",
#         "Feature_538",
#         "Feature_619",
#     ],
#     [
#         "Feature_3",
#         "Feature_4",
#         "Feature_46",
#         "Feature_111",
#         "Feature_127",
#         "Feature_179",
#         "Feature_188",
#         "Feature_215",
#         "Feature_241",
#         "Feature_281",
#         "Feature_325",
#         "Feature_326",
#         "Feature_404",
#         "Feature_478",
#         "Feature_498",
#         "Feature_544",
#         "Feature_572",
#         "Feature_577",
#         "Feature_619",
#         "Feature_653",
#         "Feature_690",
#         "Feature_694",
#         "Feature_709",
#         "Feature_711",
#         "Feature_715",
#     ],
#     [
#         "Feature_2",
#         "Feature_88",
#         "Feature_178",
#         "Feature_180",
#         "Feature_212",
#         "Feature_215",
#         "Feature_229",
#         "Feature_237",
#         "Feature_241",
#         "Feature_416",
#         "Feature_425",
#         "Feature_452",
#         "Feature_467",
#         "Feature_473",
#         "Feature_474",
#         "Feature_560",
#         "Feature_586",
#         "Feature_634",
#         "Feature_653",
#         "Feature_689",
#         "Feature_709",
#         "Feature_711",
#         "Feature_715",
#     ],
#     [
#         "Feature_4",
#         "Feature_15",
#         "Feature_177",
#         "Feature_237",
#         "Feature_394",
#         "Feature_433",
#         "Feature_453",
#         "Feature_474",
#         "Feature_477",
#         "Feature_478",
#         "Feature_498",
#         "Feature_689",
#         "Feature_690",
#         "Feature_711",
#     ],
#     [
#         "Feature_88",
#         "Feature_111",
#         "Feature_119",
#         "Feature_179",
#         "Feature_180",
#         "Feature_215",
#         "Feature_241",
#         "Feature_339",
#         "Feature_345",
#         "Feature_356",
#         "Feature_409",
#         "Feature_414",
#         "Feature_449",
#         "Feature_452",
#         "Feature_453",
#         "Feature_473",
#         "Feature_474",
#         "Feature_486",
#         "Feature_593",
#         "Feature_597",
#         "Feature_689",
#         "Feature_694",
#         "Feature_715",
#     ],
#     [
#         "Feature_164",
#         "Feature_180",
#         "Feature_230",
#         "Feature_235",
#         "Feature_237",
#         "Feature_241",
#         "Feature_345",
#         "Feature_364",
#         "Feature_452",
#         "Feature_489",
#         "Feature_563",
#         "Feature_581",
#         "Feature_654",
#         "Feature_706",
#         "Feature_715",
#     ],
#     [
#         "Feature_4",
#         "Feature_89",
#         "Feature_166",
#         "Feature_212",
#         "Feature_235",
#         "Feature_241",
#         "Feature_251",
#         "Feature_326",
#         "Feature_353",
#         "Feature_448",
#         "Feature_467",
#         "Feature_478",
#         "Feature_518",
#         "Feature_710",
#         "Feature_714",
#         "Feature_715",
#     ],
# ]

In [3]:
# # Log Reg
# top_features_list = [['Feature_1',
#   'Feature_2',
#   'Feature_4',
#   'Feature_5',
#   'Feature_6',
#   'Feature_7',
#   'Feature_8',
#   'Feature_9',
#   'Feature_10',
#   'Feature_11',
#   'Feature_12',
#   'Feature_14',
#   'Feature_15',
#   'Feature_16',
#   'Feature_17',
#   'Feature_18',
#   'Feature_19',
#   'Feature_20',
#   'Feature_21',
#   'Feature_23',
#   'Feature_24',
#   'Feature_25',
#   'Feature_28',
#   'Feature_29',
#   'Feature_30',
#   'Feature_31',
#   'Feature_32',
#   'Feature_36',
#   'Feature_39',
#   'Feature_41',
#   'Feature_42',
#   'Feature_43',
#   'Feature_45',
#   'Feature_46',
#   'Feature_47',
#   'Feature_49',
#   'Feature_50',
#   'Feature_51',
#   'Feature_53',
#   'Feature_54',
#   'Feature_58',
#   'Feature_59',
#   'Feature_61',
#   'Feature_62',
#   'Feature_63',
#   'Feature_65',
#   'Feature_66',
#   'Feature_67',
#   'Feature_69',
#   'Feature_70',
#   'Feature_72',
#   'Feature_74',
#   'Feature_76',
#   'Feature_79',
#   'Feature_81',
#   'Feature_84',
#   'Feature_85',
#   'Feature_86',
#   'Feature_87',
#   'Feature_88',
#   'Feature_89',
#   'Feature_91',
#   'Feature_92',
#   'Feature_93',
#   'Feature_95',
#   'Feature_100',
#   'Feature_102',
#   'Feature_104',
#   'Feature_105',
#   'Feature_107',
#   'Feature_108',
#   'Feature_109',
#   'Feature_114',
#   'Feature_116',
#   'Feature_117',
#   'Feature_118',
#   'Feature_120',
#   'Feature_121',
#   'Feature_123',
#   'Feature_124',
#   'Feature_125',
#   'Feature_127',
#   'Feature_128',
#   'Feature_130',
#   'Feature_131',
#   'Feature_132',
#   'Feature_136',
#   'Feature_137',
#   'Feature_138',
#   'Feature_139',
#   'Feature_140',
#   'Feature_141',
#   'Feature_142',
#   'Feature_144',
#   'Feature_145',
#   'Feature_146',
#   'Feature_147',
#   'Feature_149',
#   'Feature_150',
#   'Feature_151',
#   'Feature_154',
#   'Feature_155',
#   'Feature_156',
#   'Feature_157',
#   'Feature_158',
#   'Feature_159',
#   'Feature_161',
#   'Feature_162',
#   'Feature_163',
#   'Feature_166',
#   'Feature_168',
#   'Feature_171',
#   'Feature_172',
#   'Feature_174',
#   'Feature_175',
#   'Feature_176',
#   'Feature_178',
#   'Feature_180',
#   'Feature_181',
#   'Feature_182',
#   'Feature_184',
#   'Feature_188',
#   'Feature_189',
#   'Feature_190',
#   'Feature_192',
#   'Feature_195',
#   'Feature_196',
#   'Feature_197',
#   'Feature_198',
#   'Feature_199',
#   'Feature_200',
#   'Feature_201',
#   'Feature_202',
#   'Feature_203',
#   'Feature_205',
#   'Feature_206',
#   'Feature_207',
#   'Feature_210',
#   'Feature_211',
#   'Feature_212',
#   'Feature_214',
#   'Feature_216',
#   'Feature_218',
#   'Feature_220',
#   'Feature_221',
#   'Feature_222',
#   'Feature_223',
#   'Feature_225',
#   'Feature_226',
#   'Feature_227',
#   'Feature_228',
#   'Feature_231',
#   'Feature_233',
#   'Feature_235',
#   'Feature_236',
#   'Feature_237',
#   'Feature_240',
#   'Feature_241',
#   'Feature_242',
#   'Feature_243',
#   'Feature_246',
#   'Feature_249',
#   'Feature_250',
#   'Feature_251',
#   'Feature_252',
#   'Feature_254',
#   'Feature_255',
#   'Feature_258',
#   'Feature_260',
#   'Feature_261',
#   'Feature_262',
#   'Feature_265',
#   'Feature_266',
#   'Feature_276',
#   'Feature_277',
#   'Feature_278',
#   'Feature_279',
#   'Feature_281',
#   'Feature_283',
#   'Feature_284',
#   'Feature_285',
#   'Feature_286',
#   'Feature_288',
#   'Feature_289',
#   'Feature_290',
#   'Feature_291',
#   'Feature_296',
#   'Feature_298',
#   'Feature_299',
#   'Feature_300',
#   'Feature_303',
#   'Feature_304',
#   'Feature_305',
#   'Feature_306',
#   'Feature_307',
#   'Feature_308',
#   'Feature_311',
#   'Feature_314',
#   'Feature_318',
#   'Feature_321',
#   'Feature_322',
#   'Feature_323',
#   'Feature_325',
#   'Feature_326',
#   'Feature_327',
#   'Feature_328',
#   'Feature_329',
#   'Feature_330',
#   'Feature_331',
#   'Feature_332',
#   'Feature_333',
#   'Feature_335',
#   'Feature_336',
#   'Feature_337',
#   'Feature_341',
#   'Feature_342',
#   'Feature_343',
#   'Feature_344',
#   'Feature_345',
#   'Feature_346',
#   'Feature_347',
#   'Feature_348',
#   'Feature_349',
#   'Feature_350',
#   'Feature_351',
#   'Feature_352',
#   'Feature_355',
#   'Feature_360',
#   'Feature_361',
#   'Feature_364',
#   'Feature_365',
#   'Feature_367',
#   'Feature_374',
#   'Feature_375',
#   'Feature_377',
#   'Feature_378',
#   'Feature_379',
#   'Feature_380',
#   'Feature_381',
#   'Feature_383',
#   'Feature_384',
#   'Feature_389',
#   'Feature_391',
#   'Feature_392',
#   'Feature_393',
#   'Feature_394',
#   'Feature_395',
#   'Feature_396',
#   'Feature_398',
#   'Feature_399',
#   'Feature_400',
#   'Feature_401',
#   'Feature_402',
#   'Feature_404',
#   'Feature_408',
#   'Feature_410',
#   'Feature_411',
#   'Feature_412',
#   'Feature_413',
#   'Feature_414',
#   'Feature_416',
#   'Feature_417',
#   'Feature_418',
#   'Feature_421',
#   'Feature_425',
#   'Feature_426',
#   'Feature_427',
#   'Feature_429',
#   'Feature_432',
#   'Feature_433',
#   'Feature_434',
#   'Feature_436',
#   'Feature_440',
#   'Feature_442',
#   'Feature_443',
#   'Feature_444',
#   'Feature_445',
#   'Feature_446',
#   'Feature_448',
#   'Feature_449',
#   'Feature_451',
#   'Feature_453',
#   'Feature_455',
#   'Feature_457',
#   'Feature_458',
#   'Feature_459',
#   'Feature_461',
#   'Feature_463',
#   'Feature_465',
#   'Feature_468',
#   'Feature_470',
#   'Feature_472',
#   'Feature_473',
#   'Feature_474',
#   'Feature_477',
#   'Feature_479',
#   'Feature_480',
#   'Feature_483',
#   'Feature_486',
#   'Feature_487',
#   'Feature_488',
#   'Feature_489',
#   'Feature_490',
#   'Feature_495',
#   'Feature_497',
#   'Feature_498',
#   'Feature_499',
#   'Feature_502',
#   'Feature_503',
#   'Feature_504',
#   'Feature_505',
#   'Feature_506',
#   'Feature_507',
#   'Feature_510',
#   'Feature_512',
#   'Feature_513',
#   'Feature_515',
#   'Feature_516',
#   'Feature_517',
#   'Feature_518',
#   'Feature_519',
#   'Feature_522',
#   'Feature_523',
#   'Feature_525',
#   'Feature_527',
#   'Feature_528',
#   'Feature_529',
#   'Feature_533',
#   'Feature_534',
#   'Feature_537',
#   'Feature_539',
#   'Feature_540',
#   'Feature_542',
#   'Feature_544',
#   'Feature_545',
#   'Feature_546',
#   'Feature_548',
#   'Feature_550',
#   'Feature_552',
#   'Feature_555',
#   'Feature_556',
#   'Feature_557',
#   'Feature_559',
#   'Feature_560',
#   'Feature_562',
#   'Feature_563',
#   'Feature_564',
#   'Feature_565',
#   'Feature_566',
#   'Feature_568',
#   'Feature_569',
#   'Feature_570',
#   'Feature_571',
#   'Feature_575',
#   'Feature_576',
#   'Feature_577',
#   'Feature_578',
#   'Feature_580',
#   'Feature_582',
#   'Feature_583',
#   'Feature_584',
#   'Feature_585',
#   'Feature_587',
#   'Feature_588',
#   'Feature_590',
#   'Feature_591',
#   'Feature_592',
#   'Feature_593',
#   'Feature_595',
#   'Feature_597',
#   'Feature_598',
#   'Feature_599',
#   'Feature_600',
#   'Feature_601',
#   'Feature_602',
#   'Feature_603',
#   'Feature_604',
#   'Feature_608',
#   'Feature_609',
#   'Feature_611',
#   'Feature_616',
#   'Feature_619',
#   'Feature_620',
#   'Feature_621',
#   'Feature_625',
#   'Feature_629',
#   'Feature_630',
#   'Feature_631',
#   'Feature_632',
#   'Feature_633',
#   'Feature_634',
#   'Feature_635',
#   'Feature_637',
#   'Feature_638',
#   'Feature_641',
#   'Feature_642',
#   'Feature_643',
#   'Feature_644',
#   'Feature_646',
#   'Feature_647',
#   'Feature_648',
#   'Feature_649',
#   'Feature_651',
#   'Feature_652',
#   'Feature_653',
#   'Feature_654',
#   'Feature_655',
#   'Feature_656',
#   'Feature_657',
#   'Feature_659',
#   'Feature_662',
#   'Feature_663',
#   'Feature_664',
#   'Feature_666',
#   'Feature_668',
#   'Feature_669',
#   'Feature_670',
#   'Feature_671',
#   'Feature_673',
#   'Feature_676',
#   'Feature_677',
#   'Feature_678',
#   'Feature_682',
#   'Feature_683',
#   'Feature_684',
#   'Feature_685',
#   'Feature_686',
#   'Feature_688',
#   'Feature_690',
#   'Feature_692',
#   'Feature_694',
#   'Feature_695',
#   'Feature_696',
#   'Feature_699',
#   'Feature_701',
#   'Feature_702',
#   'Feature_705',
#   'Feature_707',
#   'Feature_708',
#   'Feature_709',
#   'Feature_710',
#   'Feature_711',
#   'Feature_714',
#   'Feature_715'],
#  ['Feature_1',
#   'Feature_2',
#   'Feature_3',
#   'Feature_4',
#   'Feature_5',
#   'Feature_8',
#   'Feature_9',
#   'Feature_12',
#   'Feature_19',
#   'Feature_20',
#   'Feature_21',
#   'Feature_22',
#   'Feature_26',
#   'Feature_27',
#   'Feature_30',
#   'Feature_31',
#   'Feature_32',
#   'Feature_33',
#   'Feature_34',
#   'Feature_35',
#   'Feature_36',
#   'Feature_38',
#   'Feature_39',
#   'Feature_41',
#   'Feature_42',
#   'Feature_43',
#   'Feature_44',
#   'Feature_46',
#   'Feature_47',
#   'Feature_48',
#   'Feature_54',
#   'Feature_56',
#   'Feature_58',
#   'Feature_60',
#   'Feature_63',
#   'Feature_65',
#   'Feature_67',
#   'Feature_68',
#   'Feature_70',
#   'Feature_71',
#   'Feature_73',
#   'Feature_75',
#   'Feature_79',
#   'Feature_87',
#   'Feature_90',
#   'Feature_93',
#   'Feature_95',
#   'Feature_96',
#   'Feature_100',
#   'Feature_101',
#   'Feature_104',
#   'Feature_106',
#   'Feature_107',
#   'Feature_108',
#   'Feature_109',
#   'Feature_112',
#   'Feature_114',
#   'Feature_115',
#   'Feature_116',
#   'Feature_118',
#   'Feature_119',
#   'Feature_121',
#   'Feature_124',
#   'Feature_127',
#   'Feature_129',
#   'Feature_131',
#   'Feature_132',
#   'Feature_133',
#   'Feature_134',
#   'Feature_138',
#   'Feature_142',
#   'Feature_143',
#   'Feature_144',
#   'Feature_145',
#   'Feature_146',
#   'Feature_151',
#   'Feature_154',
#   'Feature_160',
#   'Feature_163',
#   'Feature_164',
#   'Feature_165',
#   'Feature_166',
#   'Feature_171',
#   'Feature_172',
#   'Feature_173',
#   'Feature_179',
#   'Feature_180',
#   'Feature_182',
#   'Feature_186',
#   'Feature_187',
#   'Feature_190',
#   'Feature_191',
#   'Feature_192',
#   'Feature_194',
#   'Feature_199',
#   'Feature_200',
#   'Feature_201',
#   'Feature_206',
#   'Feature_209',
#   'Feature_210',
#   'Feature_211',
#   'Feature_213',
#   'Feature_216',
#   'Feature_221',
#   'Feature_222',
#   'Feature_225',
#   'Feature_227',
#   'Feature_228',
#   'Feature_231',
#   'Feature_233',
#   'Feature_235',
#   'Feature_236',
#   'Feature_237',
#   'Feature_241',
#   'Feature_242',
#   'Feature_245',
#   'Feature_246',
#   'Feature_247',
#   'Feature_248',
#   'Feature_249',
#   'Feature_251',
#   'Feature_256',
#   'Feature_257',
#   'Feature_258',
#   'Feature_271',
#   'Feature_272',
#   'Feature_273',
#   'Feature_275',
#   'Feature_276',
#   'Feature_277',
#   'Feature_278',
#   'Feature_280',
#   'Feature_281',
#   'Feature_283',
#   'Feature_284',
#   'Feature_287',
#   'Feature_290',
#   'Feature_291',
#   'Feature_292',
#   'Feature_294',
#   'Feature_302',
#   'Feature_303',
#   'Feature_304',
#   'Feature_306',
#   'Feature_308',
#   'Feature_312',
#   'Feature_314',
#   'Feature_324',
#   'Feature_326',
#   'Feature_327',
#   'Feature_330',
#   'Feature_331',
#   'Feature_332',
#   'Feature_333',
#   'Feature_335',
#   'Feature_337',
#   'Feature_338',
#   'Feature_341',
#   'Feature_342',
#   'Feature_344',
#   'Feature_345',
#   'Feature_346',
#   'Feature_347',
#   'Feature_348',
#   'Feature_349',
#   'Feature_350',
#   'Feature_351',
#   'Feature_355',
#   'Feature_356',
#   'Feature_358',
#   'Feature_359',
#   'Feature_361',
#   'Feature_362',
#   'Feature_364',
#   'Feature_365',
#   'Feature_368',
#   'Feature_369',
#   'Feature_374',
#   'Feature_375',
#   'Feature_377',
#   'Feature_378',
#   'Feature_380',
#   'Feature_382',
#   'Feature_384',
#   'Feature_385',
#   'Feature_386',
#   'Feature_387',
#   'Feature_388',
#   'Feature_389',
#   'Feature_390',
#   'Feature_394',
#   'Feature_397',
#   'Feature_400',
#   'Feature_401',
#   'Feature_404',
#   'Feature_409',
#   'Feature_411',
#   'Feature_413',
#   'Feature_416',
#   'Feature_417',
#   'Feature_423',
#   'Feature_424',
#   'Feature_427',
#   'Feature_434',
#   'Feature_435',
#   'Feature_437',
#   'Feature_438',
#   'Feature_439',
#   'Feature_443',
#   'Feature_444',
#   'Feature_446',
#   'Feature_447',
#   'Feature_448',
#   'Feature_450',
#   'Feature_453',
#   'Feature_458',
#   'Feature_459',
#   'Feature_462',
#   'Feature_464',
#   'Feature_468',
#   'Feature_470',
#   'Feature_472',
#   'Feature_473',
#   'Feature_474',
#   'Feature_477',
#   'Feature_481',
#   'Feature_482',
#   'Feature_483',
#   'Feature_484',
#   'Feature_485',
#   'Feature_487',
#   'Feature_495',
#   'Feature_496',
#   'Feature_498',
#   'Feature_504',
#   'Feature_505',
#   'Feature_508',
#   'Feature_509',
#   'Feature_512',
#   'Feature_513',
#   'Feature_514',
#   'Feature_517',
#   'Feature_518',
#   'Feature_519',
#   'Feature_520',
#   'Feature_521',
#   'Feature_528',
#   'Feature_529',
#   'Feature_531',
#   'Feature_533',
#   'Feature_534',
#   'Feature_535',
#   'Feature_536',
#   'Feature_537',
#   'Feature_538',
#   'Feature_539',
#   'Feature_540',
#   'Feature_541',
#   'Feature_542',
#   'Feature_546',
#   'Feature_549',
#   'Feature_550',
#   'Feature_552',
#   'Feature_555',
#   'Feature_556',
#   'Feature_557',
#   'Feature_561',
#   'Feature_563',
#   'Feature_564',
#   'Feature_567',
#   'Feature_572',
#   'Feature_574',
#   'Feature_576',
#   'Feature_581',
#   'Feature_582',
#   'Feature_583',
#   'Feature_584',
#   'Feature_585',
#   'Feature_586',
#   'Feature_592',
#   'Feature_593',
#   'Feature_594',
#   'Feature_596',
#   'Feature_597',
#   'Feature_598',
#   'Feature_601',
#   'Feature_603',
#   'Feature_604',
#   'Feature_605',
#   'Feature_606',
#   'Feature_607',
#   'Feature_608',
#   'Feature_609',
#   'Feature_610',
#   'Feature_612',
#   'Feature_613',
#   'Feature_614',
#   'Feature_615',
#   'Feature_617',
#   'Feature_621',
#   'Feature_622',
#   'Feature_623',
#   'Feature_624',
#   'Feature_626',
#   'Feature_631',
#   'Feature_634',
#   'Feature_635',
#   'Feature_636',
#   'Feature_637',
#   'Feature_638',
#   'Feature_641',
#   'Feature_642',
#   'Feature_651',
#   'Feature_652',
#   'Feature_653',
#   'Feature_656',
#   'Feature_661',
#   'Feature_663',
#   'Feature_664',
#   'Feature_673',
#   'Feature_674',
#   'Feature_675',
#   'Feature_676',
#   'Feature_678',
#   'Feature_680',
#   'Feature_681',
#   'Feature_683',
#   'Feature_684',
#   'Feature_685',
#   'Feature_687',
#   'Feature_690',
#   'Feature_695',
#   'Feature_696',
#   'Feature_699',
#   'Feature_705',
#   'Feature_707',
#   'Feature_708',
#   'Feature_709',
#   'Feature_714'],
#  ['Feature_1',
#   'Feature_2',
#   'Feature_3',
#   'Feature_5',
#   'Feature_6',
#   'Feature_8',
#   'Feature_9',
#   'Feature_10',
#   'Feature_11',
#   'Feature_12',
#   'Feature_13',
#   'Feature_15',
#   'Feature_19',
#   'Feature_20',
#   'Feature_21',
#   'Feature_22',
#   'Feature_23',
#   'Feature_26',
#   'Feature_27',
#   'Feature_28',
#   'Feature_29',
#   'Feature_30',
#   'Feature_31',
#   'Feature_32',
#   'Feature_33',
#   'Feature_34',
#   'Feature_38',
#   'Feature_39',
#   'Feature_41',
#   'Feature_43',
#   'Feature_44',
#   'Feature_45',
#   'Feature_46',
#   'Feature_48',
#   'Feature_50',
#   'Feature_52',
#   'Feature_53',
#   'Feature_56',
#   'Feature_57',
#   'Feature_58',
#   'Feature_60',
#   'Feature_63',
#   'Feature_64',
#   'Feature_65',
#   'Feature_66',
#   'Feature_67',
#   'Feature_68',
#   'Feature_69',
#   'Feature_70',
#   'Feature_71',
#   'Feature_73',
#   'Feature_77',
#   'Feature_78',
#   'Feature_79',
#   'Feature_83',
#   'Feature_84',
#   'Feature_86',
#   'Feature_87',
#   'Feature_90',
#   'Feature_92',
#   'Feature_93',
#   'Feature_94',
#   'Feature_99',
#   'Feature_100',
#   'Feature_101',
#   'Feature_103',
#   'Feature_108',
#   'Feature_110',
#   'Feature_112',
#   'Feature_113',
#   'Feature_114',
#   'Feature_118',
#   'Feature_120',
#   'Feature_121',
#   'Feature_123',
#   'Feature_124',
#   'Feature_128',
#   'Feature_129',
#   'Feature_130',
#   'Feature_131',
#   'Feature_132',
#   'Feature_133',
#   'Feature_138',
#   'Feature_139',
#   'Feature_144',
#   'Feature_146',
#   'Feature_147',
#   'Feature_151',
#   'Feature_157',
#   'Feature_160',
#   'Feature_162',
#   'Feature_163',
#   'Feature_165',
#   'Feature_166',
#   'Feature_167',
#   'Feature_168',
#   'Feature_170',
#   'Feature_171',
#   'Feature_175',
#   'Feature_178',
#   'Feature_180',
#   'Feature_181',
#   'Feature_186',
#   'Feature_187',
#   'Feature_188',
#   'Feature_189',
#   'Feature_190',
#   'Feature_192',
#   'Feature_202',
#   'Feature_203',
#   'Feature_204',
#   'Feature_206',
#   'Feature_207',
#   'Feature_209',
#   'Feature_212',
#   'Feature_214',
#   'Feature_216',
#   'Feature_218',
#   'Feature_219',
#   'Feature_220',
#   'Feature_221',
#   'Feature_223',
#   'Feature_228',
#   'Feature_229',
#   'Feature_231',
#   'Feature_236',
#   'Feature_237',
#   'Feature_240',
#   'Feature_241',
#   'Feature_242',
#   'Feature_243',
#   'Feature_245',
#   'Feature_246',
#   'Feature_249',
#   'Feature_252',
#   'Feature_254',
#   'Feature_255',
#   'Feature_259',
#   'Feature_260',
#   'Feature_262',
#   'Feature_263',
#   'Feature_264',
#   'Feature_265',
#   'Feature_266',
#   'Feature_267',
#   'Feature_268',
#   'Feature_269',
#   'Feature_270',
#   'Feature_273',
#   'Feature_277',
#   'Feature_278',
#   'Feature_279',
#   'Feature_280',
#   'Feature_281',
#   'Feature_283',
#   'Feature_284',
#   'Feature_286',
#   'Feature_288',
#   'Feature_289',
#   'Feature_290',
#   'Feature_294',
#   'Feature_295',
#   'Feature_299',
#   'Feature_300',
#   'Feature_301',
#   'Feature_303',
#   'Feature_304',
#   'Feature_305',
#   'Feature_306',
#   'Feature_308',
#   'Feature_310',
#   'Feature_311',
#   'Feature_314',
#   'Feature_315',
#   'Feature_316',
#   'Feature_320',
#   'Feature_321',
#   'Feature_323',
#   'Feature_327',
#   'Feature_339',
#   'Feature_345',
#   'Feature_347',
#   'Feature_348',
#   'Feature_349',
#   'Feature_350',
#   'Feature_355',
#   'Feature_356',
#   'Feature_358',
#   'Feature_360',
#   'Feature_361',
#   'Feature_362',
#   'Feature_366',
#   'Feature_367',
#   'Feature_368',
#   'Feature_369',
#   'Feature_372',
#   'Feature_373',
#   'Feature_374',
#   'Feature_375',
#   'Feature_376',
#   'Feature_379',
#   'Feature_381',
#   'Feature_383',
#   'Feature_384',
#   'Feature_388',
#   'Feature_390',
#   'Feature_394',
#   'Feature_401',
#   'Feature_402',
#   'Feature_406',
#   'Feature_407',
#   'Feature_408',
#   'Feature_412',
#   'Feature_415',
#   'Feature_417',
#   'Feature_423',
#   'Feature_426',
#   'Feature_427',
#   'Feature_429',
#   'Feature_436',
#   'Feature_437',
#   'Feature_438',
#   'Feature_439',
#   'Feature_443',
#   'Feature_444',
#   'Feature_446',
#   'Feature_449',
#   'Feature_451',
#   'Feature_453',
#   'Feature_455',
#   'Feature_456',
#   'Feature_457',
#   'Feature_458',
#   'Feature_460',
#   'Feature_463',
#   'Feature_464',
#   'Feature_465',
#   'Feature_466',
#   'Feature_468',
#   'Feature_471',
#   'Feature_474',
#   'Feature_477',
#   'Feature_478',
#   'Feature_482',
#   'Feature_486',
#   'Feature_487',
#   'Feature_489',
#   'Feature_491',
#   'Feature_492',
#   'Feature_496',
#   'Feature_497',
#   'Feature_499',
#   'Feature_504',
#   'Feature_506',
#   'Feature_507',
#   'Feature_508',
#   'Feature_515',
#   'Feature_516',
#   'Feature_517',
#   'Feature_518',
#   'Feature_519',
#   'Feature_520',
#   'Feature_521',
#   'Feature_522',
#   'Feature_523',
#   'Feature_524',
#   'Feature_526',
#   'Feature_530',
#   'Feature_532',
#   'Feature_536',
#   'Feature_537',
#   'Feature_538',
#   'Feature_539',
#   'Feature_540',
#   'Feature_541',
#   'Feature_542',
#   'Feature_544',
#   'Feature_546',
#   'Feature_550',
#   'Feature_552',
#   'Feature_553',
#   'Feature_557',
#   'Feature_558',
#   'Feature_560',
#   'Feature_563',
#   'Feature_564',
#   'Feature_567',
#   'Feature_569',
#   'Feature_579',
#   'Feature_581',
#   'Feature_582',
#   'Feature_584',
#   'Feature_585',
#   'Feature_587',
#   'Feature_588',
#   'Feature_594',
#   'Feature_595',
#   'Feature_596',
#   'Feature_598',
#   'Feature_599',
#   'Feature_603',
#   'Feature_609',
#   'Feature_612',
#   'Feature_614',
#   'Feature_615',
#   'Feature_616',
#   'Feature_618',
#   'Feature_619',
#   'Feature_620',
#   'Feature_621',
#   'Feature_622',
#   'Feature_625',
#   'Feature_627',
#   'Feature_631',
#   'Feature_632',
#   'Feature_633',
#   'Feature_634',
#   'Feature_636',
#   'Feature_637',
#   'Feature_638',
#   'Feature_639',
#   'Feature_640',
#   'Feature_642',
#   'Feature_647',
#   'Feature_648',
#   'Feature_649',
#   'Feature_652',
#   'Feature_653',
#   'Feature_663',
#   'Feature_664',
#   'Feature_666',
#   'Feature_668',
#   'Feature_674',
#   'Feature_675',
#   'Feature_676',
#   'Feature_680',
#   'Feature_681',
#   'Feature_682',
#   'Feature_683',
#   'Feature_686',
#   'Feature_688',
#   'Feature_690',
#   'Feature_692',
#   'Feature_693',
#   'Feature_694',
#   'Feature_695',
#   'Feature_700',
#   'Feature_702',
#   'Feature_703',
#   'Feature_705',
#   'Feature_708',
#   'Feature_710',
#   'Feature_711',
#   'Feature_714',
#   'Feature_715'],
#  ['Feature_1',
#   'Feature_2',
#   'Feature_3',
#   'Feature_4',
#   'Feature_5',
#   'Feature_6',
#   'Feature_7',
#   'Feature_8',
#   'Feature_10',
#   'Feature_11',
#   'Feature_12',
#   'Feature_13',
#   'Feature_14',
#   'Feature_15',
#   'Feature_16',
#   'Feature_17',
#   'Feature_18',
#   'Feature_19',
#   'Feature_20',
#   'Feature_21',
#   'Feature_22',
#   'Feature_23',
#   'Feature_24',
#   'Feature_25',
#   'Feature_26',
#   'Feature_27',
#   'Feature_28',
#   'Feature_29',
#   'Feature_30',
#   'Feature_32',
#   'Feature_33',
#   'Feature_34',
#   'Feature_35',
#   'Feature_36',
#   'Feature_38',
#   'Feature_40',
#   'Feature_41',
#   'Feature_42',
#   'Feature_43',
#   'Feature_45',
#   'Feature_46',
#   'Feature_47',
#   'Feature_48',
#   'Feature_49',
#   'Feature_50',
#   'Feature_51',
#   'Feature_52',
#   'Feature_53',
#   'Feature_54',
#   'Feature_55',
#   'Feature_56',
#   'Feature_57',
#   'Feature_58',
#   'Feature_59',
#   'Feature_60',
#   'Feature_61',
#   'Feature_62',
#   'Feature_63',
#   'Feature_64',
#   'Feature_65',
#   'Feature_66',
#   'Feature_67',
#   'Feature_69',
#   'Feature_70',
#   'Feature_71',
#   'Feature_72',
#   'Feature_73',
#   'Feature_75',
#   'Feature_76',
#   'Feature_77',
#   'Feature_79',
#   'Feature_80',
#   'Feature_81',
#   'Feature_82',
#   'Feature_83',
#   'Feature_84',
#   'Feature_85',
#   'Feature_86',
#   'Feature_87',
#   'Feature_88',
#   'Feature_89',
#   'Feature_90',
#   'Feature_91',
#   'Feature_92',
#   'Feature_93',
#   'Feature_96',
#   'Feature_97',
#   'Feature_98',
#   'Feature_99',
#   'Feature_100',
#   'Feature_101',
#   'Feature_102',
#   'Feature_103',
#   'Feature_104',
#   'Feature_105',
#   'Feature_106',
#   'Feature_107',
#   'Feature_108',
#   'Feature_109',
#   'Feature_110',
#   'Feature_111',
#   'Feature_112',
#   'Feature_113',
#   'Feature_114',
#   'Feature_115',
#   'Feature_116',
#   'Feature_117',
#   'Feature_119',
#   'Feature_120',
#   'Feature_121',
#   'Feature_122',
#   'Feature_123',
#   'Feature_124',
#   'Feature_125',
#   'Feature_126',
#   'Feature_127',
#   'Feature_128',
#   'Feature_129',
#   'Feature_130',
#   'Feature_131',
#   'Feature_132',
#   'Feature_133',
#   'Feature_134',
#   'Feature_135',
#   'Feature_136',
#   'Feature_137',
#   'Feature_138',
#   'Feature_139',
#   'Feature_140',
#   'Feature_141',
#   'Feature_142',
#   'Feature_143',
#   'Feature_144',
#   'Feature_145',
#   'Feature_147',
#   'Feature_148',
#   'Feature_149',
#   'Feature_150',
#   'Feature_151',
#   'Feature_152',
#   'Feature_153',
#   'Feature_154',
#   'Feature_155',
#   'Feature_156',
#   'Feature_157',
#   'Feature_160',
#   'Feature_161',
#   'Feature_162',
#   'Feature_163',
#   'Feature_164',
#   'Feature_165',
#   'Feature_166',
#   'Feature_167',
#   'Feature_168',
#   'Feature_169',
#   'Feature_170',
#   'Feature_171',
#   'Feature_172',
#   'Feature_173',
#   'Feature_174',
#   'Feature_175',
#   'Feature_176',
#   'Feature_177',
#   'Feature_178',
#   'Feature_179',
#   'Feature_180',
#   'Feature_181',
#   'Feature_183',
#   'Feature_184',
#   'Feature_185',
#   'Feature_186',
#   'Feature_187',
#   'Feature_188',
#   'Feature_189',
#   'Feature_190',
#   'Feature_192',
#   'Feature_194',
#   'Feature_195',
#   'Feature_196',
#   'Feature_197',
#   'Feature_198',
#   'Feature_199',
#   'Feature_200',
#   'Feature_201',
#   'Feature_202',
#   'Feature_203',
#   'Feature_204',
#   'Feature_205',
#   'Feature_206',
#   'Feature_207',
#   'Feature_208',
#   'Feature_209',
#   'Feature_210',
#   'Feature_211',
#   'Feature_214',
#   'Feature_215',
#   'Feature_218',
#   'Feature_221',
#   'Feature_222',
#   'Feature_225',
#   'Feature_227',
#   'Feature_229',
#   'Feature_230',
#   'Feature_231',
#   'Feature_232',
#   'Feature_236',
#   'Feature_240',
#   'Feature_241',
#   'Feature_242',
#   'Feature_243',
#   'Feature_244',
#   'Feature_245',
#   'Feature_246',
#   'Feature_247',
#   'Feature_248',
#   'Feature_249',
#   'Feature_250',
#   'Feature_251',
#   'Feature_252',
#   'Feature_253',
#   'Feature_254',
#   'Feature_255',
#   'Feature_256',
#   'Feature_257',
#   'Feature_258',
#   'Feature_260',
#   'Feature_261',
#   'Feature_262',
#   'Feature_263',
#   'Feature_264',
#   'Feature_265',
#   'Feature_266',
#   'Feature_267',
#   'Feature_268',
#   'Feature_270',
#   'Feature_271',
#   'Feature_272',
#   'Feature_273',
#   'Feature_275',
#   'Feature_277',
#   'Feature_278',
#   'Feature_279',
#   'Feature_281',
#   'Feature_283',
#   'Feature_284',
#   'Feature_286',
#   'Feature_287',
#   'Feature_288',
#   'Feature_289',
#   'Feature_290',
#   'Feature_291',
#   'Feature_293',
#   'Feature_294',
#   'Feature_295',
#   'Feature_296',
#   'Feature_297',
#   'Feature_298',
#   'Feature_299',
#   'Feature_300',
#   'Feature_301',
#   'Feature_302',
#   'Feature_303',
#   'Feature_304',
#   'Feature_305',
#   'Feature_306',
#   'Feature_307',
#   'Feature_308',
#   'Feature_309',
#   'Feature_310',
#   'Feature_312',
#   'Feature_313',
#   'Feature_314',
#   'Feature_315',
#   'Feature_316',
#   'Feature_317',
#   'Feature_318',
#   'Feature_319',
#   'Feature_320',
#   'Feature_321',
#   'Feature_322',
#   'Feature_323',
#   'Feature_325',
#   'Feature_326',
#   'Feature_327',
#   'Feature_328',
#   'Feature_329',
#   'Feature_330',
#   'Feature_332',
#   'Feature_333',
#   'Feature_334',
#   'Feature_335',
#   'Feature_337',
#   'Feature_338',
#   'Feature_340',
#   'Feature_341',
#   'Feature_343',
#   'Feature_344',
#   'Feature_345',
#   'Feature_346',
#   'Feature_347',
#   'Feature_348',
#   'Feature_349',
#   'Feature_350',
#   'Feature_351',
#   'Feature_352',
#   'Feature_353',
#   'Feature_354',
#   'Feature_357',
#   'Feature_358',
#   'Feature_359',
#   'Feature_361',
#   'Feature_362',
#   'Feature_364',
#   'Feature_365',
#   'Feature_366',
#   'Feature_367',
#   'Feature_368',
#   'Feature_369',
#   'Feature_370',
#   'Feature_371',
#   'Feature_372',
#   'Feature_373',
#   'Feature_374',
#   'Feature_375',
#   'Feature_376',
#   'Feature_377',
#   'Feature_378',
#   'Feature_379',
#   'Feature_381',
#   'Feature_382',
#   'Feature_385',
#   'Feature_386',
#   'Feature_387',
#   'Feature_388',
#   'Feature_389',
#   'Feature_390',
#   'Feature_391',
#   'Feature_392',
#   'Feature_393',
#   'Feature_394',
#   'Feature_397',
#   'Feature_398',
#   'Feature_399',
#   'Feature_400',
#   'Feature_401',
#   'Feature_402',
#   'Feature_403',
#   'Feature_405',
#   'Feature_406',
#   'Feature_407',
#   'Feature_409',
#   'Feature_410',
#   'Feature_411',
#   'Feature_412',
#   'Feature_413',
#   'Feature_414',
#   'Feature_415',
#   'Feature_416',
#   'Feature_417',
#   'Feature_418',
#   'Feature_419',
#   'Feature_420',
#   'Feature_421',
#   'Feature_422',
#   'Feature_423',
#   'Feature_424',
#   'Feature_425',
#   'Feature_426',
#   'Feature_429',
#   'Feature_431',
#   'Feature_432',
#   'Feature_433',
#   'Feature_434',
#   'Feature_435',
#   'Feature_436',
#   'Feature_437',
#   'Feature_438',
#   'Feature_439',
#   'Feature_441',
#   'Feature_442',
#   'Feature_443',
#   'Feature_444',
#   'Feature_445',
#   'Feature_446',
#   'Feature_447',
#   'Feature_448',
#   'Feature_451',
#   'Feature_452',
#   'Feature_455',
#   'Feature_458',
#   'Feature_459',
#   'Feature_460',
#   'Feature_461',
#   'Feature_462',
#   'Feature_463',
#   'Feature_464',
#   'Feature_465',
#   'Feature_466',
#   'Feature_467',
#   'Feature_468',
#   'Feature_469',
#   'Feature_471',
#   'Feature_477',
#   'Feature_478',
#   'Feature_479',
#   'Feature_480',
#   'Feature_481',
#   'Feature_482',
#   'Feature_483',
#   'Feature_484',
#   'Feature_485',
#   'Feature_487',
#   'Feature_488',
#   'Feature_489',
#   'Feature_490',
#   'Feature_491',
#   'Feature_492',
#   'Feature_493',
#   'Feature_494',
#   'Feature_495',
#   'Feature_496',
#   'Feature_497',
#   'Feature_498',
#   'Feature_499',
#   'Feature_500',
#   'Feature_501',
#   'Feature_502',
#   'Feature_503',
#   'Feature_504',
#   'Feature_507',
#   'Feature_508',
#   'Feature_510',
#   'Feature_512',
#   'Feature_514',
#   'Feature_515',
#   'Feature_516',
#   'Feature_518',
#   'Feature_519',
#   'Feature_521',
#   'Feature_522',
#   'Feature_524',
#   'Feature_525',
#   'Feature_526',
#   'Feature_527',
#   'Feature_528',
#   'Feature_529',
#   'Feature_530',
#   'Feature_531',
#   'Feature_533',
#   'Feature_534',
#   'Feature_535',
#   'Feature_536',
#   'Feature_537',
#   'Feature_538',
#   'Feature_539',
#   'Feature_540',
#   'Feature_541',
#   'Feature_542',
#   'Feature_543',
#   'Feature_544',
#   'Feature_546',
#   'Feature_547',
#   'Feature_549',
#   'Feature_550',
#   'Feature_551',
#   'Feature_552',
#   'Feature_555',
#   'Feature_556',
#   'Feature_557',
#   'Feature_558',
#   'Feature_559',
#   'Feature_560',
#   'Feature_562',
#   'Feature_563',
#   'Feature_564',
#   'Feature_565',
#   'Feature_566',
#   'Feature_567',
#   'Feature_568',
#   'Feature_569',
#   'Feature_570',
#   'Feature_571',
#   'Feature_572',
#   'Feature_573',
#   'Feature_574',
#   'Feature_575',
#   'Feature_576',
#   'Feature_577',
#   'Feature_578',
#   'Feature_579',
#   'Feature_580',
#   'Feature_581',
#   'Feature_582',
#   'Feature_583',
#   'Feature_584',
#   'Feature_585',
#   'Feature_586',
#   'Feature_587',
#   'Feature_588',
#   'Feature_589',
#   'Feature_590',
#   'Feature_591',
#   'Feature_592',
#   'Feature_595',
#   'Feature_596',
#   'Feature_597',
#   'Feature_598',
#   'Feature_599',
#   'Feature_601',
#   'Feature_602',
#   'Feature_603',
#   'Feature_604',
#   'Feature_607',
#   'Feature_608',
#   'Feature_609',
#   'Feature_610',
#   'Feature_611',
#   'Feature_613',
#   'Feature_614',
#   'Feature_615',
#   'Feature_616',
#   'Feature_617',
#   'Feature_618',
#   'Feature_619',
#   'Feature_621',
#   'Feature_622',
#   'Feature_623',
#   'Feature_624',
#   'Feature_625',
#   'Feature_626',
#   'Feature_627',
#   'Feature_628',
#   'Feature_629',
#   'Feature_630',
#   'Feature_631',
#   'Feature_632',
#   'Feature_633',
#   'Feature_634',
#   'Feature_635',
#   'Feature_636',
#   'Feature_638',
#   'Feature_639',
#   'Feature_640',
#   'Feature_641',
#   'Feature_642',
#   'Feature_643',
#   'Feature_644',
#   'Feature_645',
#   'Feature_646',
#   'Feature_647',
#   'Feature_648',
#   'Feature_649',
#   'Feature_650',
#   'Feature_651',
#   'Feature_652',
#   'Feature_653',
#   'Feature_654',
#   'Feature_655',
#   'Feature_656',
#   'Feature_657',
#   'Feature_658',
#   'Feature_659',
#   'Feature_661',
#   'Feature_662',
#   'Feature_663',
#   'Feature_664',
#   'Feature_665',
#   'Feature_666',
#   'Feature_667',
#   'Feature_668',
#   'Feature_670',
#   'Feature_671',
#   'Feature_672',
#   'Feature_673',
#   'Feature_674',
#   'Feature_675',
#   'Feature_677',
#   'Feature_678',
#   'Feature_679',
#   'Feature_680',
#   'Feature_681',
#   'Feature_682',
#   'Feature_683',
#   'Feature_684',
#   'Feature_685',
#   'Feature_688',
#   'Feature_689',
#   'Feature_692',
#   'Feature_695',
#   'Feature_696',
#   'Feature_697',
#   'Feature_698',
#   'Feature_699',
#   'Feature_700',
#   'Feature_701',
#   'Feature_702',
#   'Feature_703',
#   'Feature_704',
#   'Feature_705',
#   'Feature_706',
#   'Feature_708',
#   'Feature_709',
#   'Feature_710',
#   'Feature_715'],
#  ['Feature_1',
#   'Feature_2',
#   'Feature_3',
#   'Feature_4',
#   'Feature_5',
#   'Feature_6',
#   'Feature_7',
#   'Feature_9',
#   'Feature_10',
#   'Feature_11',
#   'Feature_12',
#   'Feature_13',
#   'Feature_14',
#   'Feature_15',
#   'Feature_16',
#   'Feature_19',
#   'Feature_20',
#   'Feature_23',
#   'Feature_24',
#   'Feature_25',
#   'Feature_26',
#   'Feature_28',
#   'Feature_29',
#   'Feature_30',
#   'Feature_31',
#   'Feature_32',
#   'Feature_33',
#   'Feature_34',
#   'Feature_35',
#   'Feature_36',
#   'Feature_38',
#   'Feature_40',
#   'Feature_41',
#   'Feature_42',
#   'Feature_43',
#   'Feature_44',
#   'Feature_45',
#   'Feature_46',
#   'Feature_47',
#   'Feature_48',
#   'Feature_49',
#   'Feature_50',
#   'Feature_51',
#   'Feature_52',
#   'Feature_53',
#   'Feature_54',
#   'Feature_55',
#   'Feature_56',
#   'Feature_57',
#   'Feature_58',
#   'Feature_60',
#   'Feature_61',
#   'Feature_64',
#   'Feature_65',
#   'Feature_66',
#   'Feature_67',
#   'Feature_68',
#   'Feature_70',
#   'Feature_72',
#   'Feature_75',
#   'Feature_77',
#   'Feature_78',
#   'Feature_79',
#   'Feature_81',
#   'Feature_82',
#   'Feature_83',
#   'Feature_84',
#   'Feature_85',
#   'Feature_86',
#   'Feature_87',
#   'Feature_88',
#   'Feature_89',
#   'Feature_90',
#   'Feature_92',
#   'Feature_93',
#   'Feature_95',
#   'Feature_96',
#   'Feature_98',
#   'Feature_99',
#   'Feature_101',
#   'Feature_103',
#   'Feature_104',
#   'Feature_106',
#   'Feature_107',
#   'Feature_108',
#   'Feature_109',
#   'Feature_110',
#   'Feature_111',
#   'Feature_112',
#   'Feature_113',
#   'Feature_117',
#   'Feature_118',
#   'Feature_119',
#   'Feature_120',
#   'Feature_121',
#   'Feature_122',
#   'Feature_123',
#   'Feature_124',
#   'Feature_125',
#   'Feature_126',
#   'Feature_127',
#   'Feature_128',
#   'Feature_129',
#   'Feature_130',
#   'Feature_131',
#   'Feature_132',
#   'Feature_133',
#   'Feature_136',
#   'Feature_138',
#   'Feature_139',
#   'Feature_142',
#   'Feature_143',
#   'Feature_144',
#   'Feature_145',
#   'Feature_146',
#   'Feature_147',
#   'Feature_148',
#   'Feature_149',
#   'Feature_150',
#   'Feature_151',
#   'Feature_153',
#   'Feature_154',
#   'Feature_156',
#   'Feature_158',
#   'Feature_159',
#   'Feature_160',
#   'Feature_161',
#   'Feature_162',
#   'Feature_163',
#   'Feature_164',
#   'Feature_166',
#   'Feature_167',
#   'Feature_168',
#   'Feature_171',
#   'Feature_172',
#   'Feature_173',
#   'Feature_174',
#   'Feature_177',
#   'Feature_178',
#   'Feature_179',
#   'Feature_180',
#   'Feature_181',
#   'Feature_184',
#   'Feature_185',
#   'Feature_186',
#   'Feature_187',
#   'Feature_188',
#   'Feature_189',
#   'Feature_190',
#   'Feature_192',
#   'Feature_193',
#   'Feature_194',
#   'Feature_195',
#   'Feature_196',
#   'Feature_197',
#   'Feature_198',
#   'Feature_199',
#   'Feature_203',
#   'Feature_208',
#   'Feature_209',
#   'Feature_210',
#   'Feature_211',
#   'Feature_212',
#   'Feature_213',
#   'Feature_214',
#   'Feature_215',
#   'Feature_216',
#   'Feature_217',
#   'Feature_219',
#   'Feature_220',
#   'Feature_221',
#   'Feature_222',
#   'Feature_225',
#   'Feature_226',
#   'Feature_227',
#   'Feature_228',
#   'Feature_229',
#   'Feature_230',
#   'Feature_232',
#   'Feature_233',
#   'Feature_235',
#   'Feature_236',
#   'Feature_237',
#   'Feature_240',
#   'Feature_241',
#   'Feature_242',
#   'Feature_243',
#   'Feature_244',
#   'Feature_245',
#   'Feature_249',
#   'Feature_250',
#   'Feature_251',
#   'Feature_256',
#   'Feature_257',
#   'Feature_260',
#   'Feature_261',
#   'Feature_262',
#   'Feature_263',
#   'Feature_264',
#   'Feature_265',
#   'Feature_266',
#   'Feature_267',
#   'Feature_268',
#   'Feature_270',
#   'Feature_271',
#   'Feature_272',
#   'Feature_273',
#   'Feature_275',
#   'Feature_277',
#   'Feature_278',
#   'Feature_280',
#   'Feature_282',
#   'Feature_283',
#   'Feature_284',
#   'Feature_287',
#   'Feature_288',
#   'Feature_289',
#   'Feature_290',
#   'Feature_292',
#   'Feature_293',
#   'Feature_294',
#   'Feature_296',
#   'Feature_297',
#   'Feature_299',
#   'Feature_301',
#   'Feature_302',
#   'Feature_303',
#   'Feature_304',
#   'Feature_305',
#   'Feature_306',
#   'Feature_307',
#   'Feature_309',
#   'Feature_311',
#   'Feature_312',
#   'Feature_314',
#   'Feature_315',
#   'Feature_318',
#   'Feature_319',
#   'Feature_320',
#   'Feature_321',
#   'Feature_322',
#   'Feature_323',
#   'Feature_324',
#   'Feature_325',
#   'Feature_326',
#   'Feature_327',
#   'Feature_328',
#   'Feature_329',
#   'Feature_330',
#   'Feature_331',
#   'Feature_332',
#   'Feature_333',
#   'Feature_334',
#   'Feature_335',
#   'Feature_337',
#   'Feature_338',
#   'Feature_339',
#   'Feature_340',
#   'Feature_342',
#   'Feature_343',
#   'Feature_344',
#   'Feature_345',
#   'Feature_346',
#   'Feature_348',
#   'Feature_349',
#   'Feature_350',
#   'Feature_352',
#   'Feature_353',
#   'Feature_354',
#   'Feature_356',
#   'Feature_357',
#   'Feature_359',
#   'Feature_361',
#   'Feature_363',
#   'Feature_364',
#   'Feature_365',
#   'Feature_366',
#   'Feature_367',
#   'Feature_368',
#   'Feature_369',
#   'Feature_370',
#   'Feature_371',
#   'Feature_372',
#   'Feature_373',
#   'Feature_375',
#   'Feature_376',
#   'Feature_377',
#   'Feature_378',
#   'Feature_379',
#   'Feature_380',
#   'Feature_382',
#   'Feature_383',
#   'Feature_384',
#   'Feature_385',
#   'Feature_386',
#   'Feature_387',
#   'Feature_388',
#   'Feature_389',
#   'Feature_390',
#   'Feature_391',
#   'Feature_392',
#   'Feature_393',
#   'Feature_394',
#   'Feature_395',
#   'Feature_396',
#   'Feature_397',
#   'Feature_398',
#   'Feature_400',
#   'Feature_401',
#   'Feature_402',
#   'Feature_403',
#   'Feature_405',
#   'Feature_406',
#   'Feature_407',
#   'Feature_408',
#   'Feature_409',
#   'Feature_411',
#   'Feature_412',
#   'Feature_414',
#   'Feature_415',
#   'Feature_416',
#   'Feature_417',
#   'Feature_419',
#   'Feature_421',
#   'Feature_422',
#   'Feature_424',
#   'Feature_425',
#   'Feature_426',
#   'Feature_427',
#   'Feature_429',
#   'Feature_430',
#   'Feature_431',
#   'Feature_433',
#   'Feature_434',
#   'Feature_435',
#   'Feature_436',
#   'Feature_437',
#   'Feature_438',
#   'Feature_440',
#   'Feature_444',
#   'Feature_445',
#   'Feature_446',
#   'Feature_447',
#   'Feature_448',
#   'Feature_449',
#   'Feature_450',
#   'Feature_451',
#   'Feature_452',
#   'Feature_453',
#   'Feature_454',
#   'Feature_456',
#   'Feature_457',
#   'Feature_458',
#   'Feature_459',
#   'Feature_461',
#   'Feature_462',
#   'Feature_463',
#   'Feature_464',
#   'Feature_465',
#   'Feature_466',
#   'Feature_467',
#   'Feature_469',
#   'Feature_470',
#   'Feature_472',
#   'Feature_473',
#   'Feature_474',
#   'Feature_477',
#   'Feature_478',
#   'Feature_479',
#   'Feature_480',
#   'Feature_481',
#   'Feature_482',
#   'Feature_484',
#   'Feature_485',
#   'Feature_486',
#   'Feature_488',
#   'Feature_489',
#   'Feature_490',
#   'Feature_495',
#   'Feature_496',
#   'Feature_497',
#   'Feature_498',
#   'Feature_499',
#   'Feature_502',
#   'Feature_503',
#   'Feature_504',
#   'Feature_505',
#   'Feature_506',
#   'Feature_507',
#   'Feature_508',
#   'Feature_510',
#   'Feature_512',
#   'Feature_514',
#   'Feature_515',
#   'Feature_516',
#   'Feature_517',
#   'Feature_518',
#   'Feature_519',
#   'Feature_521',
#   'Feature_522',
#   'Feature_523',
#   'Feature_524',
#   'Feature_525',
#   'Feature_526',
#   'Feature_527',
#   'Feature_528',
#   'Feature_529',
#   'Feature_530',
#   'Feature_531',
#   'Feature_532',
#   'Feature_533',
#   'Feature_534',
#   'Feature_536',
#   'Feature_538',
#   'Feature_539',
#   'Feature_541',
#   'Feature_543',
#   'Feature_544',
#   'Feature_545',
#   'Feature_546',
#   'Feature_548',
#   'Feature_549',
#   'Feature_550',
#   'Feature_551',
#   'Feature_552',
#   'Feature_555',
#   'Feature_556',
#   'Feature_557',
#   'Feature_558',
#   'Feature_559',
#   'Feature_560',
#   'Feature_561',
#   'Feature_562',
#   'Feature_563',
#   'Feature_564',
#   'Feature_566',
#   'Feature_568',
#   'Feature_570',
#   'Feature_571',
#   'Feature_572',
#   'Feature_573',
#   'Feature_574',
#   'Feature_575',
#   'Feature_576',
#   'Feature_577',
#   'Feature_578',
#   'Feature_579',
#   'Feature_580',
#   'Feature_581',
#   'Feature_582',
#   'Feature_583',
#   'Feature_584',
#   'Feature_585',
#   'Feature_586',
#   'Feature_587',
#   'Feature_589',
#   'Feature_590',
#   'Feature_591',
#   'Feature_592',
#   'Feature_593',
#   'Feature_594',
#   'Feature_595',
#   'Feature_596',
#   'Feature_597',
#   'Feature_598',
#   'Feature_599',
#   'Feature_600',
#   'Feature_601',
#   'Feature_602',
#   'Feature_603',
#   'Feature_604',
#   'Feature_605',
#   'Feature_606',
#   'Feature_607',
#   'Feature_608',
#   'Feature_610',
#   'Feature_611',
#   'Feature_612',
#   'Feature_613',
#   'Feature_614',
#   'Feature_615',
#   'Feature_616',
#   'Feature_617',
#   'Feature_618',
#   'Feature_619',
#   'Feature_620',
#   'Feature_621',
#   'Feature_622',
#   'Feature_623',
#   'Feature_624',
#   'Feature_626',
#   'Feature_627',
#   'Feature_631',
#   'Feature_632',
#   'Feature_633',
#   'Feature_635',
#   'Feature_637',
#   'Feature_638',
#   'Feature_640',
#   'Feature_641',
#   'Feature_642',
#   'Feature_643',
#   'Feature_645',
#   'Feature_646',
#   'Feature_647',
#   'Feature_648',
#   'Feature_651',
#   'Feature_652',
#   'Feature_654',
#   'Feature_658',
#   'Feature_659',
#   'Feature_660',
#   'Feature_661',
#   'Feature_662',
#   'Feature_663',
#   'Feature_664',
#   'Feature_667',
#   'Feature_668',
#   'Feature_669',
#   'Feature_670',
#   'Feature_671',
#   'Feature_672',
#   'Feature_673',
#   'Feature_674',
#   'Feature_675',
#   'Feature_676',
#   'Feature_677',
#   'Feature_680',
#   'Feature_681',
#   'Feature_682',
#   'Feature_683',
#   'Feature_685',
#   'Feature_686',
#   'Feature_687',
#   'Feature_688',
#   'Feature_689',
#   'Feature_690',
#   'Feature_691',
#   'Feature_693',
#   'Feature_694',
#   'Feature_695',
#   'Feature_696',
#   'Feature_697',
#   'Feature_700',
#   'Feature_702',
#   'Feature_703',
#   'Feature_704',
#   'Feature_706',
#   'Feature_707',
#   'Feature_708',
#   'Feature_709',
#   'Feature_710',
#   'Feature_711',
#   'Feature_714',
#   'Feature_715'],
#  ['Feature_1',
#   'Feature_2',
#   'Feature_3',
#   'Feature_4',
#   'Feature_5',
#   'Feature_7',
#   'Feature_9',
#   'Feature_12',
#   'Feature_14',
#   'Feature_15',
#   'Feature_18',
#   'Feature_19',
#   'Feature_20',
#   'Feature_21',
#   'Feature_22',
#   'Feature_23',
#   'Feature_24',
#   'Feature_25',
#   'Feature_26',
#   'Feature_30',
#   'Feature_32',
#   'Feature_33',
#   'Feature_34',
#   'Feature_35',
#   'Feature_36',
#   'Feature_38',
#   'Feature_39',
#   'Feature_40',
#   'Feature_41',
#   'Feature_42',
#   'Feature_44',
#   'Feature_45',
#   'Feature_46',
#   'Feature_47',
#   'Feature_48',
#   'Feature_49',
#   'Feature_50',
#   'Feature_51',
#   'Feature_52',
#   'Feature_54',
#   'Feature_55',
#   'Feature_56',
#   'Feature_57',
#   'Feature_58',
#   'Feature_59',
#   'Feature_60',
#   'Feature_61',
#   'Feature_62',
#   'Feature_64',
#   'Feature_65',
#   'Feature_66',
#   'Feature_67',
#   'Feature_69',
#   'Feature_70',
#   'Feature_71',
#   'Feature_72',
#   'Feature_74',
#   'Feature_76',
#   'Feature_77',
#   'Feature_78',
#   'Feature_79',
#   'Feature_81',
#   'Feature_82',
#   'Feature_85',
#   'Feature_86',
#   'Feature_89',
#   'Feature_90',
#   'Feature_91',
#   'Feature_92',
#   'Feature_94',
#   'Feature_95',
#   'Feature_96',
#   'Feature_97',
#   'Feature_98',
#   'Feature_100',
#   'Feature_101',
#   'Feature_103',
#   'Feature_104',
#   'Feature_105',
#   'Feature_106',
#   'Feature_108',
#   'Feature_109',
#   'Feature_110',
#   'Feature_112',
#   'Feature_113',
#   'Feature_114',
#   'Feature_115',
#   'Feature_116',
#   'Feature_117',
#   'Feature_118',
#   'Feature_119',
#   'Feature_120',
#   'Feature_122',
#   'Feature_124',
#   'Feature_125',
#   'Feature_126',
#   'Feature_127',
#   'Feature_128',
#   'Feature_129',
#   'Feature_130',
#   'Feature_133',
#   'Feature_134',
#   'Feature_135',
#   'Feature_136',
#   'Feature_140',
#   'Feature_141',
#   'Feature_143',
#   'Feature_144',
#   'Feature_145',
#   'Feature_147',
#   'Feature_148',
#   'Feature_149',
#   'Feature_150',
#   'Feature_153',
#   'Feature_154',
#   'Feature_156',
#   'Feature_157',
#   'Feature_158',
#   'Feature_159',
#   'Feature_160',
#   'Feature_163',
#   'Feature_165',
#   'Feature_166',
#   'Feature_167',
#   'Feature_168',
#   'Feature_169',
#   'Feature_170',
#   'Feature_171',
#   'Feature_172',
#   'Feature_174',
#   'Feature_175',
#   'Feature_176',
#   'Feature_177',
#   'Feature_178',
#   'Feature_181',
#   'Feature_182',
#   'Feature_183',
#   'Feature_185',
#   'Feature_186',
#   'Feature_187',
#   'Feature_189',
#   'Feature_190',
#   'Feature_194',
#   'Feature_195',
#   'Feature_196',
#   'Feature_197',
#   'Feature_199',
#   'Feature_202',
#   'Feature_205',
#   'Feature_206',
#   'Feature_208',
#   'Feature_209',
#   'Feature_210',
#   'Feature_212',
#   'Feature_213',
#   'Feature_215',
#   'Feature_217',
#   'Feature_220',
#   'Feature_223',
#   'Feature_224',
#   'Feature_225',
#   'Feature_231',
#   'Feature_232',
#   'Feature_233',
#   'Feature_235',
#   'Feature_236',
#   'Feature_237',
#   'Feature_240',
#   'Feature_241',
#   'Feature_242',
#   'Feature_244',
#   'Feature_245',
#   'Feature_246',
#   'Feature_247',
#   'Feature_248',
#   'Feature_249',
#   'Feature_252',
#   'Feature_254',
#   'Feature_255',
#   'Feature_256',
#   'Feature_257',
#   'Feature_258',
#   'Feature_260',
#   'Feature_261',
#   'Feature_262',
#   'Feature_264',
#   'Feature_267',
#   'Feature_268',
#   'Feature_269',
#   'Feature_271',
#   'Feature_272',
#   'Feature_273',
#   'Feature_275',
#   'Feature_276',
#   'Feature_277',
#   'Feature_278',
#   'Feature_279',
#   'Feature_280',
#   'Feature_281',
#   'Feature_282',
#   'Feature_283',
#   'Feature_284',
#   'Feature_285',
#   'Feature_286',
#   'Feature_287',
#   'Feature_288',
#   'Feature_290',
#   'Feature_291',
#   'Feature_293',
#   'Feature_294',
#   'Feature_295',
#   'Feature_296',
#   'Feature_297',
#   'Feature_299',
#   'Feature_300',
#   'Feature_301',
#   'Feature_302',
#   'Feature_303',
#   'Feature_304',
#   'Feature_306',
#   'Feature_307',
#   'Feature_308',
#   'Feature_310',
#   'Feature_311',
#   'Feature_312',
#   'Feature_315',
#   'Feature_316',
#   'Feature_317',
#   'Feature_318',
#   'Feature_319',
#   'Feature_320',
#   'Feature_321',
#   'Feature_324',
#   'Feature_325',
#   'Feature_326',
#   'Feature_327',
#   'Feature_329',
#   'Feature_330',
#   'Feature_331',
#   'Feature_332',
#   'Feature_333',
#   'Feature_334',
#   'Feature_336',
#   'Feature_337',
#   'Feature_339',
#   'Feature_340',
#   'Feature_341',
#   'Feature_342',
#   'Feature_343',
#   'Feature_345',
#   'Feature_346',
#   'Feature_347',
#   'Feature_349',
#   'Feature_350',
#   'Feature_351',
#   'Feature_352',
#   'Feature_353',
#   'Feature_354',
#   'Feature_355',
#   'Feature_356',
#   'Feature_357',
#   'Feature_359',
#   'Feature_360',
#   'Feature_361',
#   'Feature_362',
#   'Feature_363',
#   'Feature_365',
#   'Feature_366',
#   'Feature_367',
#   'Feature_368',
#   'Feature_369',
#   'Feature_371',
#   'Feature_372',
#   'Feature_373',
#   'Feature_374',
#   'Feature_375',
#   'Feature_376',
#   'Feature_380',
#   'Feature_381',
#   'Feature_382',
#   'Feature_383',
#   'Feature_385',
#   'Feature_388',
#   'Feature_389',
#   'Feature_390',
#   'Feature_391',
#   'Feature_393',
#   'Feature_394',
#   'Feature_395',
#   'Feature_396',
#   'Feature_397',
#   'Feature_398',
#   'Feature_399',
#   'Feature_401',
#   'Feature_402',
#   'Feature_403',
#   'Feature_404',
#   'Feature_405',
#   'Feature_406',
#   'Feature_407',
#   'Feature_408',
#   'Feature_411',
#   'Feature_412',
#   'Feature_413',
#   'Feature_414',
#   'Feature_415',
#   'Feature_416',
#   'Feature_417',
#   'Feature_418',
#   'Feature_420',
#   'Feature_421',
#   'Feature_422',
#   'Feature_423',
#   'Feature_424',
#   'Feature_426',
#   'Feature_427',
#   'Feature_430',
#   'Feature_431',
#   'Feature_432',
#   'Feature_433',
#   'Feature_434',
#   'Feature_435',
#   'Feature_436',
#   'Feature_443',
#   'Feature_444',
#   'Feature_447',
#   'Feature_449',
#   'Feature_450',
#   'Feature_452',
#   'Feature_454',
#   'Feature_457',
#   'Feature_461',
#   'Feature_464',
#   'Feature_468',
#   'Feature_469',
#   'Feature_470',
#   'Feature_471',
#   'Feature_472',
#   'Feature_473',
#   'Feature_474',
#   'Feature_477',
#   'Feature_478',
#   'Feature_479',
#   'Feature_481',
#   'Feature_482',
#   'Feature_483',
#   'Feature_486',
#   'Feature_487',
#   'Feature_488',
#   'Feature_489',
#   'Feature_490',
#   'Feature_491',
#   'Feature_492',
#   'Feature_493',
#   'Feature_494',
#   'Feature_495',
#   'Feature_497',
#   'Feature_498',
#   'Feature_500',
#   'Feature_501',
#   'Feature_502',
#   'Feature_503',
#   'Feature_504',
#   'Feature_508',
#   'Feature_510',
#   'Feature_512',
#   'Feature_514',
#   'Feature_515',
#   'Feature_517',
#   'Feature_518',
#   'Feature_519',
#   'Feature_520',
#   'Feature_521',
#   'Feature_523',
#   'Feature_524',
#   'Feature_525',
#   'Feature_526',
#   'Feature_527',
#   'Feature_529',
#   'Feature_531',
#   'Feature_532',
#   'Feature_534',
#   'Feature_535',
#   'Feature_536',
#   'Feature_540',
#   'Feature_541',
#   'Feature_542',
#   'Feature_543',
#   'Feature_544',
#   'Feature_546',
#   'Feature_548',
#   'Feature_550',
#   'Feature_551',
#   'Feature_554',
#   'Feature_555',
#   'Feature_557',
#   'Feature_558',
#   'Feature_559',
#   'Feature_560',
#   'Feature_561',
#   'Feature_562',
#   'Feature_563',
#   'Feature_564',
#   'Feature_565',
#   'Feature_566',
#   'Feature_567',
#   'Feature_568',
#   'Feature_569',
#   'Feature_570',
#   'Feature_572',
#   'Feature_573',
#   'Feature_574',
#   'Feature_575',
#   'Feature_576',
#   'Feature_577',
#   'Feature_578',
#   'Feature_579',
#   'Feature_580',
#   'Feature_581',
#   'Feature_583',
#   'Feature_585',
#   'Feature_586',
#   'Feature_587',
#   'Feature_588',
#   'Feature_589',
#   'Feature_590',
#   'Feature_595',
#   'Feature_597',
#   'Feature_598',
#   'Feature_599',
#   'Feature_600',
#   'Feature_601',
#   'Feature_602',
#   'Feature_603',
#   'Feature_604',
#   'Feature_605',
#   'Feature_606',
#   'Feature_607',
#   'Feature_608',
#   'Feature_609',
#   'Feature_610',
#   'Feature_611',
#   'Feature_612',
#   'Feature_613',
#   'Feature_614',
#   'Feature_615',
#   'Feature_616',
#   'Feature_617',
#   'Feature_619',
#   'Feature_620',
#   'Feature_621',
#   'Feature_622',
#   'Feature_623',
#   'Feature_624',
#   'Feature_625',
#   'Feature_627',
#   'Feature_628',
#   'Feature_629',
#   'Feature_630',
#   'Feature_631',
#   'Feature_632',
#   'Feature_633',
#   'Feature_634',
#   'Feature_635',
#   'Feature_637',
#   'Feature_638',
#   'Feature_639',
#   'Feature_640',
#   'Feature_641',
#   'Feature_642',
#   'Feature_643',
#   'Feature_644',
#   'Feature_645',
#   'Feature_646',
#   'Feature_647',
#   'Feature_648',
#   'Feature_649',
#   'Feature_651',
#   'Feature_652',
#   'Feature_653',
#   'Feature_654',
#   'Feature_655',
#   'Feature_657',
#   'Feature_658',
#   'Feature_659',
#   'Feature_661',
#   'Feature_663',
#   'Feature_664',
#   'Feature_667',
#   'Feature_668',
#   'Feature_670',
#   'Feature_671',
#   'Feature_672',
#   'Feature_680',
#   'Feature_684',
#   'Feature_686',
#   'Feature_687',
#   'Feature_689',
#   'Feature_691',
#   'Feature_694',
#   'Feature_697',
#   'Feature_698',
#   'Feature_699',
#   'Feature_700',
#   'Feature_702',
#   'Feature_705',
#   'Feature_706',
#   'Feature_707',
#   'Feature_708',
#   'Feature_709',
#   'Feature_710',
#   'Feature_711',
#   'Feature_714',
#   'Feature_715'],
#  ['Feature_4',
#   'Feature_6',
#   'Feature_8',
#   'Feature_11',
#   'Feature_13',
#   'Feature_14',
#   'Feature_15',
#   'Feature_17',
#   'Feature_18',
#   'Feature_21',
#   'Feature_22',
#   'Feature_23',
#   'Feature_25',
#   'Feature_26',
#   'Feature_27',
#   'Feature_30',
#   'Feature_31',
#   'Feature_32',
#   'Feature_33',
#   'Feature_40',
#   'Feature_42',
#   'Feature_43',
#   'Feature_44',
#   'Feature_45',
#   'Feature_46',
#   'Feature_49',
#   'Feature_51',
#   'Feature_53',
#   'Feature_57',
#   'Feature_59',
#   'Feature_60',
#   'Feature_64',
#   'Feature_66',
#   'Feature_68',
#   'Feature_70',
#   'Feature_72',
#   'Feature_73',
#   'Feature_74',
#   'Feature_75',
#   'Feature_76',
#   'Feature_81',
#   'Feature_82',
#   'Feature_84',
#   'Feature_86',
#   'Feature_88',
#   'Feature_90',
#   'Feature_91',
#   'Feature_93',
#   'Feature_97',
#   'Feature_98',
#   'Feature_101',
#   'Feature_102',
#   'Feature_104',
#   'Feature_105',
#   'Feature_106',
#   'Feature_107',
#   'Feature_112',
#   'Feature_113',
#   'Feature_115',
#   'Feature_118',
#   'Feature_119',
#   'Feature_122',
#   'Feature_123',
#   'Feature_125',
#   'Feature_129',
#   'Feature_132',
#   'Feature_133',
#   'Feature_135',
#   'Feature_136',
#   'Feature_137',
#   'Feature_139',
#   'Feature_143',
#   'Feature_144',
#   'Feature_149',
#   'Feature_150',
#   'Feature_153',
#   'Feature_154',
#   'Feature_155',
#   'Feature_156',
#   'Feature_159',
#   'Feature_160',
#   'Feature_166',
#   'Feature_169',
#   'Feature_170',
#   'Feature_173',
#   'Feature_174',
#   'Feature_175',
#   'Feature_176',
#   'Feature_186',
#   'Feature_188',
#   'Feature_189',
#   'Feature_192',
#   'Feature_193',
#   'Feature_194',
#   'Feature_195',
#   'Feature_196',
#   'Feature_197',
#   'Feature_198',
#   'Feature_199',
#   'Feature_202',
#   'Feature_203',
#   'Feature_204',
#   'Feature_207',
#   'Feature_209',
#   'Feature_212',
#   'Feature_213',
#   'Feature_214',
#   'Feature_215',
#   'Feature_216',
#   'Feature_217',
#   'Feature_223',
#   'Feature_225',
#   'Feature_228',
#   'Feature_229',
#   'Feature_231',
#   'Feature_232',
#   'Feature_233',
#   'Feature_235',
#   'Feature_241',
#   'Feature_243',
#   'Feature_245',
#   'Feature_246',
#   'Feature_247',
#   'Feature_248',
#   'Feature_251',
#   'Feature_253',
#   'Feature_254',
#   'Feature_255',
#   'Feature_259',
#   'Feature_260',
#   'Feature_262',
#   'Feature_263',
#   'Feature_264',
#   'Feature_268',
#   'Feature_269',
#   'Feature_270',
#   'Feature_271',
#   'Feature_275',
#   'Feature_277',
#   'Feature_278',
#   'Feature_279',
#   'Feature_280',
#   'Feature_281',
#   'Feature_282',
#   'Feature_285',
#   'Feature_286',
#   'Feature_288',
#   'Feature_290',
#   'Feature_291',
#   'Feature_294',
#   'Feature_302',
#   'Feature_303',
#   'Feature_305',
#   'Feature_306',
#   'Feature_307',
#   'Feature_308',
#   'Feature_310',
#   'Feature_312',
#   'Feature_313',
#   'Feature_315',
#   'Feature_316',
#   'Feature_318',
#   'Feature_319',
#   'Feature_321',
#   'Feature_322',
#   'Feature_323',
#   'Feature_325',
#   'Feature_327',
#   'Feature_328',
#   'Feature_331',
#   'Feature_332',
#   'Feature_333',
#   'Feature_334',
#   'Feature_335',
#   'Feature_338',
#   'Feature_339',
#   'Feature_340',
#   'Feature_342',
#   'Feature_345',
#   'Feature_348',
#   'Feature_349',
#   'Feature_350',
#   'Feature_354',
#   'Feature_355',
#   'Feature_356',
#   'Feature_358',
#   'Feature_363',
#   'Feature_365',
#   'Feature_368',
#   'Feature_369',
#   'Feature_375',
#   'Feature_376',
#   'Feature_377',
#   'Feature_378',
#   'Feature_381',
#   'Feature_384',
#   'Feature_386',
#   'Feature_387',
#   'Feature_388',
#   'Feature_389',
#   'Feature_390',
#   'Feature_393',
#   'Feature_394',
#   'Feature_397',
#   'Feature_401',
#   'Feature_402',
#   'Feature_405',
#   'Feature_406',
#   'Feature_407',
#   'Feature_411',
#   'Feature_412',
#   'Feature_413',
#   'Feature_414',
#   'Feature_416',
#   'Feature_418',
#   'Feature_420',
#   'Feature_421',
#   'Feature_423',
#   'Feature_425',
#   'Feature_426',
#   'Feature_427',
#   'Feature_430',
#   'Feature_432',
#   'Feature_433',
#   'Feature_434',
#   'Feature_436',
#   'Feature_439',
#   'Feature_444',
#   'Feature_446',
#   'Feature_447',
#   'Feature_449',
#   'Feature_450',
#   'Feature_451',
#   'Feature_452',
#   'Feature_453',
#   'Feature_454',
#   'Feature_461',
#   'Feature_463',
#   'Feature_464',
#   'Feature_465',
#   'Feature_466',
#   'Feature_468',
#   'Feature_469',
#   'Feature_470',
#   'Feature_471',
#   'Feature_472',
#   'Feature_473',
#   'Feature_478',
#   'Feature_480',
#   'Feature_481',
#   'Feature_486',
#   'Feature_487',
#   'Feature_488',
#   'Feature_490',
#   'Feature_495',
#   'Feature_497',
#   'Feature_499',
#   'Feature_500',
#   'Feature_501',
#   'Feature_502',
#   'Feature_503',
#   'Feature_505',
#   'Feature_506',
#   'Feature_508',
#   'Feature_509',
#   'Feature_510',
#   'Feature_512',
#   'Feature_515',
#   'Feature_516',
#   'Feature_517',
#   'Feature_518',
#   'Feature_519',
#   'Feature_520',
#   'Feature_521',
#   'Feature_523',
#   'Feature_524',
#   'Feature_526',
#   'Feature_527',
#   'Feature_529',
#   'Feature_530',
#   'Feature_536',
#   'Feature_538',
#   'Feature_539',
#   'Feature_540',
#   'Feature_544',
#   'Feature_545',
#   'Feature_550',
#   'Feature_551',
#   'Feature_555',
#   'Feature_556',
#   'Feature_558',
#   'Feature_559',
#   'Feature_560',
#   'Feature_561',
#   'Feature_562',
#   'Feature_564',
#   'Feature_566',
#   'Feature_567',
#   'Feature_571',
#   'Feature_572',
#   'Feature_574',
#   'Feature_575',
#   'Feature_576',
#   'Feature_577',
#   'Feature_578',
#   'Feature_579',
#   'Feature_585',
#   'Feature_586',
#   'Feature_587',
#   'Feature_589',
#   'Feature_591',
#   'Feature_592',
#   'Feature_593',
#   'Feature_595',
#   'Feature_597',
#   'Feature_599',
#   'Feature_601',
#   'Feature_602',
#   'Feature_607',
#   'Feature_610',
#   'Feature_611',
#   'Feature_612',
#   'Feature_613',
#   'Feature_614',
#   'Feature_615',
#   'Feature_616',
#   'Feature_619',
#   'Feature_620',
#   'Feature_621',
#   'Feature_623',
#   'Feature_624',
#   'Feature_625',
#   'Feature_627',
#   'Feature_628',
#   'Feature_629',
#   'Feature_630',
#   'Feature_631',
#   'Feature_632',
#   'Feature_633',
#   'Feature_634',
#   'Feature_636',
#   'Feature_637',
#   'Feature_638',
#   'Feature_640',
#   'Feature_641',
#   'Feature_642',
#   'Feature_645',
#   'Feature_648',
#   'Feature_649',
#   'Feature_650',
#   'Feature_651',
#   'Feature_652',
#   'Feature_655',
#   'Feature_660',
#   'Feature_661',
#   'Feature_663',
#   'Feature_664',
#   'Feature_665',
#   'Feature_666',
#   'Feature_668',
#   'Feature_669',
#   'Feature_670',
#   'Feature_671',
#   'Feature_672',
#   'Feature_673',
#   'Feature_677',
#   'Feature_678',
#   'Feature_679',
#   'Feature_680',
#   'Feature_682',
#   'Feature_683',
#   'Feature_684',
#   'Feature_686',
#   'Feature_687',
#   'Feature_688',
#   'Feature_689',
#   'Feature_690',
#   'Feature_691',
#   'Feature_697',
#   'Feature_703',
#   'Feature_705',
#   'Feature_706',
#   'Feature_707',
#   'Feature_708',
#   'Feature_709',
#   'Feature_710',
#   'Feature_715'],
#  ['Feature_1',
#   'Feature_2',
#   'Feature_3',
#   'Feature_4',
#   'Feature_5',
#   'Feature_6',
#   'Feature_7',
#   'Feature_8',
#   'Feature_9',
#   'Feature_10',
#   'Feature_11',
#   'Feature_12',
#   'Feature_13',
#   'Feature_14',
#   'Feature_16',
#   'Feature_17',
#   'Feature_18',
#   'Feature_19',
#   'Feature_20',
#   'Feature_21',
#   'Feature_26',
#   'Feature_27',
#   'Feature_30',
#   'Feature_31',
#   'Feature_32',
#   'Feature_33',
#   'Feature_34',
#   'Feature_38',
#   'Feature_41',
#   'Feature_42',
#   'Feature_43',
#   'Feature_44',
#   'Feature_45',
#   'Feature_46',
#   'Feature_47',
#   'Feature_48',
#   'Feature_49',
#   'Feature_54',
#   'Feature_55',
#   'Feature_57',
#   'Feature_59',
#   'Feature_60',
#   'Feature_61',
#   'Feature_64',
#   'Feature_66',
#   'Feature_67',
#   'Feature_69',
#   'Feature_70',
#   'Feature_72',
#   'Feature_73',
#   'Feature_74',
#   'Feature_75',
#   'Feature_76',
#   'Feature_77',
#   'Feature_78',
#   'Feature_79',
#   'Feature_80',
#   'Feature_81',
#   'Feature_86',
#   'Feature_87',
#   'Feature_89',
#   'Feature_94',
#   'Feature_95',
#   'Feature_96',
#   'Feature_98',
#   'Feature_100',
#   'Feature_101',
#   'Feature_103',
#   'Feature_104',
#   'Feature_105',
#   'Feature_106',
#   'Feature_107',
#   'Feature_108',
#   'Feature_109',
#   'Feature_110',
#   'Feature_111',
#   'Feature_112',
#   'Feature_113',
#   'Feature_114',
#   'Feature_115',
#   'Feature_116',
#   'Feature_117',
#   'Feature_118',
#   'Feature_119',
#   'Feature_120',
#   'Feature_121',
#   'Feature_122',
#   'Feature_123',
#   'Feature_125',
#   'Feature_126',
#   'Feature_127',
#   'Feature_128',
#   'Feature_129',
#   'Feature_131',
#   'Feature_132',
#   'Feature_133',
#   'Feature_134',
#   'Feature_135',
#   'Feature_137',
#   'Feature_138',
#   'Feature_140',
#   'Feature_141',
#   'Feature_142',
#   'Feature_145',
#   'Feature_147',
#   'Feature_148',
#   'Feature_149',
#   'Feature_150',
#   'Feature_151',
#   'Feature_152',
#   'Feature_154',
#   'Feature_155',
#   'Feature_156',
#   'Feature_157',
#   'Feature_158',
#   'Feature_159',
#   'Feature_160',
#   'Feature_161',
#   'Feature_162',
#   'Feature_164',
#   'Feature_168',
#   'Feature_169',
#   'Feature_170',
#   'Feature_171',
#   'Feature_172',
#   'Feature_173',
#   'Feature_175',
#   'Feature_176',
#   'Feature_177',
#   'Feature_178',
#   'Feature_179',
#   'Feature_180',
#   'Feature_186',
#   'Feature_187',
#   'Feature_189',
#   'Feature_190',
#   'Feature_191',
#   'Feature_192',
#   'Feature_193',
#   'Feature_194',
#   'Feature_195',
#   'Feature_196',
#   'Feature_197',
#   'Feature_198',
#   'Feature_199',
#   'Feature_200',
#   'Feature_201',
#   'Feature_202',
#   'Feature_203',
#   'Feature_204',
#   'Feature_205',
#   'Feature_206',
#   'Feature_207',
#   'Feature_208',
#   'Feature_209',
#   'Feature_210',
#   'Feature_211',
#   'Feature_213',
#   'Feature_214',
#   'Feature_215',
#   'Feature_216',
#   'Feature_220',
#   'Feature_221',
#   'Feature_222',
#   'Feature_224',
#   'Feature_225',
#   'Feature_226',
#   'Feature_227',
#   'Feature_228',
#   'Feature_229',
#   'Feature_232',
#   'Feature_233',
#   'Feature_235',
#   'Feature_236',
#   'Feature_237',
#   'Feature_240',
#   'Feature_241',
#   'Feature_243',
#   'Feature_244',
#   'Feature_245',
#   'Feature_246',
#   'Feature_247',
#   'Feature_248',
#   'Feature_251',
#   'Feature_252',
#   'Feature_253',
#   'Feature_256',
#   'Feature_257',
#   'Feature_258',
#   'Feature_259',
#   'Feature_260',
#   'Feature_262',
#   'Feature_265',
#   'Feature_266',
#   'Feature_269',
#   'Feature_271',
#   'Feature_272',
#   'Feature_273',
#   'Feature_277',
#   'Feature_278',
#   'Feature_279',
#   'Feature_280',
#   'Feature_281',
#   'Feature_283',
#   'Feature_284',
#   'Feature_285',
#   'Feature_286',
#   'Feature_287',
#   'Feature_288',
#   'Feature_290',
#   'Feature_292',
#   'Feature_294',
#   'Feature_295',
#   'Feature_296',
#   'Feature_297',
#   'Feature_299',
#   'Feature_300',
#   'Feature_301',
#   'Feature_303',
#   'Feature_306',
#   'Feature_307',
#   'Feature_308',
#   'Feature_309',
#   'Feature_310',
#   'Feature_311',
#   'Feature_312',
#   'Feature_314',
#   'Feature_315',
#   'Feature_316',
#   'Feature_317',
#   'Feature_318',
#   'Feature_319',
#   'Feature_323',
#   'Feature_324',
#   'Feature_326',
#   'Feature_327',
#   'Feature_330',
#   'Feature_331',
#   'Feature_332',
#   'Feature_334',
#   'Feature_335',
#   'Feature_336',
#   'Feature_337',
#   'Feature_338',
#   'Feature_339',
#   'Feature_340',
#   'Feature_341',
#   'Feature_343',
#   'Feature_345',
#   'Feature_346',
#   'Feature_347',
#   'Feature_348',
#   'Feature_349',
#   'Feature_350',
#   'Feature_351',
#   'Feature_352',
#   'Feature_353',
#   'Feature_354',
#   'Feature_355',
#   'Feature_356',
#   'Feature_360',
#   'Feature_363',
#   'Feature_364',
#   'Feature_366',
#   'Feature_367',
#   'Feature_368',
#   'Feature_369',
#   'Feature_370',
#   'Feature_371',
#   'Feature_372',
#   'Feature_374',
#   'Feature_375',
#   'Feature_376',
#   'Feature_377',
#   'Feature_378',
#   'Feature_382',
#   'Feature_383',
#   'Feature_384',
#   'Feature_386',
#   'Feature_387',
#   'Feature_389',
#   'Feature_390',
#   'Feature_391',
#   'Feature_392',
#   'Feature_393',
#   'Feature_394',
#   'Feature_395',
#   'Feature_396',
#   'Feature_397',
#   'Feature_398',
#   'Feature_399',
#   'Feature_400',
#   'Feature_401',
#   'Feature_405',
#   'Feature_406',
#   'Feature_407',
#   'Feature_408',
#   'Feature_409',
#   'Feature_410',
#   'Feature_411',
#   'Feature_412',
#   'Feature_413',
#   'Feature_414',
#   'Feature_415',
#   'Feature_416',
#   'Feature_417',
#   'Feature_418',
#   'Feature_423',
#   'Feature_424',
#   'Feature_426',
#   'Feature_427',
#   'Feature_432',
#   'Feature_433',
#   'Feature_434',
#   'Feature_436',
#   'Feature_438',
#   'Feature_439',
#   'Feature_441',
#   'Feature_442',
#   'Feature_443',
#   'Feature_444',
#   'Feature_445',
#   'Feature_446',
#   'Feature_447',
#   'Feature_448',
#   'Feature_450',
#   'Feature_451',
#   'Feature_452',
#   'Feature_453',
#   'Feature_457',
#   'Feature_458',
#   'Feature_459',
#   'Feature_460',
#   'Feature_461',
#   'Feature_462',
#   'Feature_464',
#   'Feature_465',
#   'Feature_466',
#   'Feature_469',
#   'Feature_470',
#   'Feature_471',
#   'Feature_472',
#   'Feature_473',
#   'Feature_474',
#   'Feature_477',
#   'Feature_478',
#   'Feature_479',
#   'Feature_480',
#   'Feature_481',
#   'Feature_482',
#   'Feature_483',
#   'Feature_484',
#   'Feature_485',
#   'Feature_486',
#   'Feature_488',
#   'Feature_489',
#   'Feature_490',
#   'Feature_491',
#   'Feature_492',
#   'Feature_493',
#   'Feature_494',
#   'Feature_495',
#   'Feature_496',
#   'Feature_497',
#   'Feature_499',
#   'Feature_500',
#   'Feature_501',
#   'Feature_502',
#   'Feature_503',
#   'Feature_504',
#   'Feature_505',
#   'Feature_506',
#   'Feature_507',
#   'Feature_509',
#   'Feature_510',
#   'Feature_514',
#   'Feature_516',
#   'Feature_517',
#   'Feature_518',
#   'Feature_520',
#   'Feature_521',
#   'Feature_523',
#   'Feature_524',
#   'Feature_526',
#   'Feature_527',
#   'Feature_528',
#   'Feature_531',
#   'Feature_532',
#   'Feature_533',
#   'Feature_534',
#   'Feature_535',
#   'Feature_538',
#   'Feature_539',
#   'Feature_540',
#   'Feature_541',
#   'Feature_542',
#   'Feature_543',
#   'Feature_544',
#   'Feature_545',
#   'Feature_546',
#   'Feature_547',
#   'Feature_548',
#   'Feature_549',
#   'Feature_550',
#   'Feature_551',
#   'Feature_552',
#   'Feature_553',
#   'Feature_554',
#   'Feature_555',
#   'Feature_556',
#   'Feature_557',
#   'Feature_559',
#   'Feature_560',
#   'Feature_561',
#   'Feature_563',
#   'Feature_564',
#   'Feature_565',
#   'Feature_566',
#   'Feature_569',
#   'Feature_570',
#   'Feature_571',
#   'Feature_572',
#   'Feature_573',
#   'Feature_575',
#   'Feature_576',
#   'Feature_577',
#   'Feature_579',
#   'Feature_580',
#   'Feature_582',
#   'Feature_583',
#   'Feature_584',
#   'Feature_585',
#   'Feature_586',
#   'Feature_587',
#   'Feature_588',
#   'Feature_589',
#   'Feature_590',
#   'Feature_593',
#   'Feature_594',
#   'Feature_595',
#   'Feature_597',
#   'Feature_598',
#   'Feature_600',
#   'Feature_603',
#   'Feature_604',
#   'Feature_607',
#   'Feature_609',
#   'Feature_611',
#   'Feature_612',
#   'Feature_613',
#   'Feature_614',
#   'Feature_615',
#   'Feature_617',
#   'Feature_618',
#   'Feature_619',
#   'Feature_620',
#   'Feature_622',
#   'Feature_625',
#   'Feature_626',
#   'Feature_628',
#   'Feature_631',
#   'Feature_632',
#   'Feature_633',
#   'Feature_634',
#   'Feature_635',
#   'Feature_636',
#   'Feature_638',
#   'Feature_642',
#   'Feature_643',
#   'Feature_644',
#   'Feature_645',
#   'Feature_647',
#   'Feature_648',
#   'Feature_650',
#   'Feature_651',
#   'Feature_652',
#   'Feature_653',
#   'Feature_654',
#   'Feature_655',
#   'Feature_660',
#   'Feature_661',
#   'Feature_662',
#   'Feature_663',
#   'Feature_664',
#   'Feature_666',
#   'Feature_667',
#   'Feature_668',
#   'Feature_669',
#   'Feature_670',
#   'Feature_671',
#   'Feature_673',
#   'Feature_676',
#   'Feature_678',
#   'Feature_680',
#   'Feature_681',
#   'Feature_682',
#   'Feature_683',
#   'Feature_684',
#   'Feature_685',
#   'Feature_687',
#   'Feature_688',
#   'Feature_689',
#   'Feature_690',
#   'Feature_694',
#   'Feature_695',
#   'Feature_696',
#   'Feature_697',
#   'Feature_698',
#   'Feature_699',
#   'Feature_700',
#   'Feature_701',
#   'Feature_702',
#   'Feature_703',
#   'Feature_706',
#   'Feature_707',
#   'Feature_708',
#   'Feature_709',
#   'Feature_715'],
#  ['Feature_1',
#   'Feature_2',
#   'Feature_3',
#   'Feature_4',
#   'Feature_5',
#   'Feature_8',
#   'Feature_12',
#   'Feature_14',
#   'Feature_16',
#   'Feature_17',
#   'Feature_18',
#   'Feature_21',
#   'Feature_25',
#   'Feature_28',
#   'Feature_29',
#   'Feature_30',
#   'Feature_32',
#   'Feature_33',
#   'Feature_34',
#   'Feature_35',
#   'Feature_38',
#   'Feature_39',
#   'Feature_40',
#   'Feature_42',
#   'Feature_43',
#   'Feature_44',
#   'Feature_45',
#   'Feature_46',
#   'Feature_47',
#   'Feature_48',
#   'Feature_50',
#   'Feature_51',
#   'Feature_52',
#   'Feature_54',
#   'Feature_55',
#   'Feature_56',
#   'Feature_57',
#   'Feature_58',
#   'Feature_59',
#   'Feature_60',
#   'Feature_61',
#   'Feature_62',
#   'Feature_63',
#   'Feature_65',
#   'Feature_67',
#   'Feature_68',
#   'Feature_71',
#   'Feature_72',
#   'Feature_75',
#   'Feature_76',
#   'Feature_77',
#   'Feature_79',
#   'Feature_80',
#   'Feature_82',
#   'Feature_86',
#   'Feature_89',
#   'Feature_90',
#   'Feature_93',
#   'Feature_94',
#   'Feature_95',
#   'Feature_99',
#   'Feature_101',
#   'Feature_103',
#   'Feature_104',
#   'Feature_105',
#   'Feature_106',
#   'Feature_108',
#   'Feature_109',
#   'Feature_111',
#   'Feature_113',
#   'Feature_114',
#   'Feature_115',
#   'Feature_116',
#   'Feature_117',
#   'Feature_118',
#   'Feature_119',
#   'Feature_120',
#   'Feature_121',
#   'Feature_122',
#   'Feature_123',
#   'Feature_124',
#   'Feature_125',
#   'Feature_126',
#   'Feature_127',
#   'Feature_131',
#   'Feature_132',
#   'Feature_133',
#   'Feature_136',
#   'Feature_138',
#   'Feature_139',
#   'Feature_140',
#   'Feature_141',
#   'Feature_142',
#   'Feature_143',
#   'Feature_144',
#   'Feature_145',
#   'Feature_146',
#   'Feature_147',
#   'Feature_148',
#   'Feature_151',
#   'Feature_152',
#   'Feature_153',
#   'Feature_154',
#   'Feature_156',
#   'Feature_157',
#   'Feature_158',
#   'Feature_159',
#   'Feature_160',
#   'Feature_161',
#   'Feature_163',
#   'Feature_165',
#   'Feature_167',
#   'Feature_169',
#   'Feature_173',
#   'Feature_174',
#   'Feature_175',
#   'Feature_176',
#   'Feature_177',
#   'Feature_180',
#   'Feature_182',
#   'Feature_183',
#   'Feature_186',
#   'Feature_188',
#   'Feature_189',
#   'Feature_190',
#   'Feature_191',
#   'Feature_193',
#   'Feature_198',
#   'Feature_200',
#   'Feature_201',
#   'Feature_203',
#   'Feature_208',
#   'Feature_209',
#   'Feature_210',
#   'Feature_211',
#   'Feature_212',
#   'Feature_216',
#   'Feature_217',
#   'Feature_218',
#   'Feature_219',
#   'Feature_221',
#   'Feature_222',
#   'Feature_224',
#   'Feature_225',
#   'Feature_226',
#   'Feature_227',
#   'Feature_231',
#   'Feature_232',
#   'Feature_237',
#   'Feature_240',
#   'Feature_241',
#   'Feature_242',
#   'Feature_243',
#   'Feature_244',
#   'Feature_245',
#   'Feature_246',
#   'Feature_247',
#   'Feature_248',
#   'Feature_251',
#   'Feature_252',
#   'Feature_253',
#   'Feature_261',
#   'Feature_262',
#   'Feature_265',
#   'Feature_266',
#   'Feature_268',
#   'Feature_269',
#   'Feature_271',
#   'Feature_272',
#   'Feature_273',
#   'Feature_275',
#   'Feature_276',
#   'Feature_277',
#   'Feature_278',
#   'Feature_280',
#   'Feature_281',
#   'Feature_282',
#   'Feature_283',
#   'Feature_284',
#   'Feature_285',
#   'Feature_286',
#   'Feature_287',
#   'Feature_288',
#   'Feature_291',
#   'Feature_292',
#   'Feature_293',
#   'Feature_294',
#   'Feature_296',
#   'Feature_297',
#   'Feature_298',
#   'Feature_300',
#   'Feature_301',
#   'Feature_302',
#   'Feature_304',
#   'Feature_305',
#   'Feature_306',
#   'Feature_308',
#   'Feature_309',
#   'Feature_310',
#   'Feature_312',
#   'Feature_314',
#   'Feature_315',
#   'Feature_316',
#   'Feature_319',
#   'Feature_323',
#   'Feature_326',
#   'Feature_330',
#   'Feature_332',
#   'Feature_333',
#   'Feature_335',
#   'Feature_336',
#   'Feature_340',
#   'Feature_341',
#   'Feature_342',
#   'Feature_343',
#   'Feature_344',
#   'Feature_345',
#   'Feature_346',
#   'Feature_347',
#   'Feature_348',
#   'Feature_351',
#   'Feature_352',
#   'Feature_353',
#   'Feature_354',
#   'Feature_355',
#   'Feature_356',
#   'Feature_357',
#   'Feature_359',
#   'Feature_360',
#   'Feature_361',
#   'Feature_363',
#   'Feature_364',
#   'Feature_366',
#   'Feature_367',
#   'Feature_368',
#   'Feature_369',
#   'Feature_370',
#   'Feature_371',
#   'Feature_375',
#   'Feature_376',
#   'Feature_379',
#   'Feature_380',
#   'Feature_384',
#   'Feature_385',
#   'Feature_386',
#   'Feature_387',
#   'Feature_388',
#   'Feature_389',
#   'Feature_392',
#   'Feature_393',
#   'Feature_394',
#   'Feature_395',
#   'Feature_396',
#   'Feature_397',
#   'Feature_398',
#   'Feature_399',
#   'Feature_402',
#   'Feature_403',
#   'Feature_404',
#   'Feature_406',
#   'Feature_407',
#   'Feature_409',
#   'Feature_410',
#   'Feature_414',
#   'Feature_415',
#   'Feature_417',
#   'Feature_419',
#   'Feature_420',
#   'Feature_421',
#   'Feature_422',
#   'Feature_423',
#   'Feature_425',
#   'Feature_426',
#   'Feature_427',
#   'Feature_429',
#   'Feature_431',
#   'Feature_434',
#   'Feature_436',
#   'Feature_439',
#   'Feature_440',
#   'Feature_443',
#   'Feature_444',
#   'Feature_446',
#   'Feature_448',
#   'Feature_449',
#   'Feature_453',
#   'Feature_454',
#   'Feature_455',
#   'Feature_456',
#   'Feature_458',
#   'Feature_459',
#   'Feature_460',
#   'Feature_461',
#   'Feature_462',
#   'Feature_463',
#   'Feature_464',
#   'Feature_465',
#   'Feature_468',
#   'Feature_469',
#   'Feature_471',
#   'Feature_474',
#   'Feature_477',
#   'Feature_478',
#   'Feature_479',
#   'Feature_480',
#   'Feature_481',
#   'Feature_482',
#   'Feature_484',
#   'Feature_485',
#   'Feature_487',
#   'Feature_488',
#   'Feature_489',
#   'Feature_490',
#   'Feature_491',
#   'Feature_492',
#   'Feature_493',
#   'Feature_494',
#   'Feature_496',
#   'Feature_498',
#   'Feature_499',
#   'Feature_500',
#   'Feature_501',
#   'Feature_505',
#   'Feature_506',
#   'Feature_507',
#   'Feature_508',
#   'Feature_509',
#   'Feature_512',
#   'Feature_513',
#   'Feature_514',
#   'Feature_515',
#   'Feature_518',
#   'Feature_519',
#   'Feature_520',
#   'Feature_521',
#   'Feature_522',
#   'Feature_523',
#   'Feature_524',
#   'Feature_525',
#   'Feature_527',
#   'Feature_528',
#   'Feature_529',
#   'Feature_530',
#   'Feature_531',
#   'Feature_532',
#   'Feature_533',
#   'Feature_534',
#   'Feature_535',
#   'Feature_536',
#   'Feature_538',
#   'Feature_539',
#   'Feature_540',
#   'Feature_541',
#   'Feature_542',
#   'Feature_545',
#   'Feature_547',
#   'Feature_548',
#   'Feature_551',
#   'Feature_553',
#   'Feature_556',
#   'Feature_559',
#   'Feature_560',
#   'Feature_561',
#   'Feature_563',
#   'Feature_564',
#   'Feature_567',
#   'Feature_568',
#   'Feature_569',
#   'Feature_570',
#   'Feature_572',
#   'Feature_573',
#   'Feature_574',
#   'Feature_575',
#   'Feature_576',
#   'Feature_578',
#   'Feature_579',
#   'Feature_580',
#   'Feature_582',
#   'Feature_583',
#   'Feature_584',
#   'Feature_587',
#   'Feature_589',
#   'Feature_590',
#   'Feature_591',
#   'Feature_592',
#   'Feature_594',
#   'Feature_595',
#   'Feature_598',
#   'Feature_599',
#   'Feature_600',
#   'Feature_602',
#   'Feature_604',
#   'Feature_607',
#   'Feature_608',
#   'Feature_610',
#   'Feature_611',
#   'Feature_612',
#   'Feature_613',
#   'Feature_614',
#   'Feature_615',
#   'Feature_617',
#   'Feature_620',
#   'Feature_621',
#   'Feature_622',
#   'Feature_623',
#   'Feature_624',
#   'Feature_625',
#   'Feature_626',
#   'Feature_627',
#   'Feature_628',
#   'Feature_631',
#   'Feature_632',
#   'Feature_633',
#   'Feature_637',
#   'Feature_639',
#   'Feature_642',
#   'Feature_644',
#   'Feature_645',
#   'Feature_646',
#   'Feature_647',
#   'Feature_648',
#   'Feature_649',
#   'Feature_650',
#   'Feature_651',
#   'Feature_652',
#   'Feature_654',
#   'Feature_655',
#   'Feature_656',
#   'Feature_657',
#   'Feature_658',
#   'Feature_659',
#   'Feature_661',
#   'Feature_662',
#   'Feature_663',
#   'Feature_664',
#   'Feature_666',
#   'Feature_668',
#   'Feature_669',
#   'Feature_671',
#   'Feature_673',
#   'Feature_677',
#   'Feature_680',
#   'Feature_681',
#   'Feature_682',
#   'Feature_683',
#   'Feature_685',
#   'Feature_686',
#   'Feature_690',
#   'Feature_691',
#   'Feature_692',
#   'Feature_693',
#   'Feature_695',
#   'Feature_696',
#   'Feature_698',
#   'Feature_699',
#   'Feature_700',
#   'Feature_705',
#   'Feature_706',
#   'Feature_708',
#   'Feature_711',
#   'Feature_715']]

In [4]:
# # XGBoost
# top_features_list = [['Feature_1',
#   'Feature_2',
#   'Feature_3',
#   'Feature_4',
#   'Feature_5',
#   'Feature_6',
#   'Feature_14',
#   'Feature_15',
#   'Feature_24',
#   'Feature_38',
#   'Feature_40',
#   'Feature_44',
#   'Feature_54',
#   'Feature_64',
#   'Feature_70',
#   'Feature_84',
#   'Feature_86',
#   'Feature_98',
#   'Feature_99',
#   'Feature_101',
#   'Feature_109',
#   'Feature_111',
#   'Feature_113',
#   'Feature_116',
#   'Feature_127',
#   'Feature_130',
#   'Feature_136',
#   'Feature_148',
#   'Feature_157',
#   'Feature_164',
#   'Feature_172',
#   'Feature_175',
#   'Feature_176',
#   'Feature_177',
#   'Feature_178',
#   'Feature_179',
#   'Feature_180',
#   'Feature_181',
#   'Feature_183',
#   'Feature_185',
#   'Feature_188',
#   'Feature_192',
#   'Feature_208',
#   'Feature_211',
#   'Feature_212',
#   'Feature_215',
#   'Feature_216',
#   'Feature_229',
#   'Feature_231',
#   'Feature_235',
#   'Feature_237',
#   'Feature_240',
#   'Feature_241',
#   'Feature_261',
#   'Feature_275',
#   'Feature_303',
#   'Feature_321',
#   'Feature_322',
#   'Feature_325',
#   'Feature_326',
#   'Feature_347',
#   'Feature_350',
#   'Feature_361',
#   'Feature_376',
#   'Feature_382',
#   'Feature_394',
#   'Feature_404',
#   'Feature_406',
#   'Feature_410',
#   'Feature_412',
#   'Feature_413',
#   'Feature_415',
#   'Feature_416',
#   'Feature_417',
#   'Feature_420',
#   'Feature_425',
#   'Feature_472',
#   'Feature_474',
#   'Feature_478',
#   'Feature_489',
#   'Feature_498',
#   'Feature_554',
#   'Feature_560',
#   'Feature_563',
#   'Feature_571',
#   'Feature_572',
#   'Feature_581',
#   'Feature_583',
#   'Feature_584',
#   'Feature_586',
#   'Feature_587',
#   'Feature_598',
#   'Feature_607',
#   'Feature_622',
#   'Feature_631',
#   'Feature_640',
#   'Feature_641',
#   'Feature_646',
#   'Feature_647',
#   'Feature_649',
#   'Feature_650',
#   'Feature_651',
#   'Feature_653',
#   'Feature_654',
#   'Feature_659',
#   'Feature_662',
#   'Feature_682',
#   'Feature_709',
#   'Feature_711',
#   'Feature_715'],
#  ['Feature_1',
#   'Feature_2',
#   'Feature_3',
#   'Feature_4',
#   'Feature_6',
#   'Feature_16',
#   'Feature_34',
#   'Feature_42',
#   'Feature_80',
#   'Feature_88',
#   'Feature_98',
#   'Feature_101',
#   'Feature_104',
#   'Feature_111',
#   'Feature_112',
#   'Feature_113',
#   'Feature_130',
#   'Feature_154',
#   'Feature_166',
#   'Feature_173',
#   'Feature_175',
#   'Feature_179',
#   'Feature_180',
#   'Feature_182',
#   'Feature_183',
#   'Feature_189',
#   'Feature_211',
#   'Feature_215',
#   'Feature_216',
#   'Feature_235',
#   'Feature_237',
#   'Feature_240',
#   'Feature_241',
#   'Feature_325',
#   'Feature_346',
#   'Feature_350',
#   'Feature_385',
#   'Feature_394',
#   'Feature_406',
#   'Feature_414',
#   'Feature_416',
#   'Feature_417',
#   'Feature_472',
#   'Feature_474',
#   'Feature_478',
#   'Feature_480',
#   'Feature_489',
#   'Feature_497',
#   'Feature_498',
#   'Feature_516',
#   'Feature_519',
#   'Feature_520',
#   'Feature_554',
#   'Feature_560',
#   'Feature_572',
#   'Feature_584',
#   'Feature_586',
#   'Feature_597',
#   'Feature_598',
#   'Feature_604',
#   'Feature_607',
#   'Feature_613',
#   'Feature_631',
#   'Feature_641',
#   'Feature_644',
#   'Feature_651',
#   'Feature_653',
#   'Feature_654',
#   'Feature_662',
#   'Feature_683',
#   'Feature_699',
#   'Feature_711',
#   'Feature_715'],
#  ['Feature_1',
#   'Feature_2',
#   'Feature_3',
#   'Feature_24',
#   'Feature_25',
#   'Feature_40',
#   'Feature_44',
#   'Feature_66',
#   'Feature_83',
#   'Feature_85',
#   'Feature_86',
#   'Feature_88',
#   'Feature_104',
#   'Feature_112',
#   'Feature_113',
#   'Feature_127',
#   'Feature_130',
#   'Feature_136',
#   'Feature_139',
#   'Feature_145',
#   'Feature_157',
#   'Feature_163',
#   'Feature_167',
#   'Feature_173',
#   'Feature_175',
#   'Feature_177',
#   'Feature_179',
#   'Feature_180',
#   'Feature_183',
#   'Feature_188',
#   'Feature_189',
#   'Feature_211',
#   'Feature_212',
#   'Feature_215',
#   'Feature_216',
#   'Feature_220',
#   'Feature_229',
#   'Feature_235',
#   'Feature_237',
#   'Feature_240',
#   'Feature_241',
#   'Feature_277',
#   'Feature_301',
#   'Feature_305',
#   'Feature_329',
#   'Feature_335',
#   'Feature_338',
#   'Feature_349',
#   'Feature_350',
#   'Feature_385',
#   'Feature_394',
#   'Feature_403',
#   'Feature_409',
#   'Feature_415',
#   'Feature_416',
#   'Feature_417',
#   'Feature_421',
#   'Feature_425',
#   'Feature_446',
#   'Feature_472',
#   'Feature_480',
#   'Feature_481',
#   'Feature_498',
#   'Feature_514',
#   'Feature_516',
#   'Feature_544',
#   'Feature_559',
#   'Feature_563',
#   'Feature_572',
#   'Feature_575',
#   'Feature_578',
#   'Feature_583',
#   'Feature_587',
#   'Feature_613',
#   'Feature_631',
#   'Feature_641',
#   'Feature_646',
#   'Feature_650',
#   'Feature_651',
#   'Feature_652',
#   'Feature_653',
#   'Feature_654',
#   'Feature_659',
#   'Feature_662',
#   'Feature_663',
#   'Feature_674',
#   'Feature_683',
#   'Feature_709',
#   'Feature_711'],
#  ['Feature_1',
#   'Feature_3',
#   'Feature_4',
#   'Feature_6',
#   'Feature_24',
#   'Feature_69',
#   'Feature_80',
#   'Feature_82',
#   'Feature_84',
#   'Feature_108',
#   'Feature_166',
#   'Feature_175',
#   'Feature_177',
#   'Feature_180',
#   'Feature_183',
#   'Feature_189',
#   'Feature_208',
#   'Feature_212',
#   'Feature_220',
#   'Feature_229',
#   'Feature_231',
#   'Feature_237',
#   'Feature_241',
#   'Feature_261',
#   'Feature_262',
#   'Feature_325',
#   'Feature_326',
#   'Feature_344',
#   'Feature_347',
#   'Feature_376',
#   'Feature_382',
#   'Feature_385',
#   'Feature_394',
#   'Feature_422',
#   'Feature_442',
#   'Feature_474',
#   'Feature_478',
#   'Feature_498',
#   'Feature_652',
#   'Feature_654',
#   'Feature_682',
#   'Feature_709',
#   'Feature_711',
#   'Feature_715'],
#  ['Feature_1',
#   'Feature_2',
#   'Feature_3',
#   'Feature_4',
#   'Feature_5',
#   'Feature_6',
#   'Feature_44',
#   'Feature_83',
#   'Feature_84',
#   'Feature_88',
#   'Feature_101',
#   'Feature_106',
#   'Feature_109',
#   'Feature_111',
#   'Feature_113',
#   'Feature_130',
#   'Feature_151',
#   'Feature_167',
#   'Feature_168',
#   'Feature_172',
#   'Feature_173',
#   'Feature_177',
#   'Feature_178',
#   'Feature_179',
#   'Feature_180',
#   'Feature_181',
#   'Feature_183',
#   'Feature_184',
#   'Feature_185',
#   'Feature_188',
#   'Feature_192',
#   'Feature_209',
#   'Feature_211',
#   'Feature_212',
#   'Feature_215',
#   'Feature_216',
#   'Feature_219',
#   'Feature_220',
#   'Feature_229',
#   'Feature_230',
#   'Feature_233',
#   'Feature_235',
#   'Feature_237',
#   'Feature_241',
#   'Feature_243',
#   'Feature_252',
#   'Feature_280',
#   'Feature_301',
#   'Feature_336',
#   'Feature_338',
#   'Feature_346',
#   'Feature_349',
#   'Feature_350',
#   'Feature_360',
#   'Feature_364',
#   'Feature_382',
#   'Feature_385',
#   'Feature_388',
#   'Feature_394',
#   'Feature_409',
#   'Feature_413',
#   'Feature_414',
#   'Feature_416',
#   'Feature_417',
#   'Feature_445',
#   'Feature_446',
#   'Feature_472',
#   'Feature_474',
#   'Feature_480',
#   'Feature_489',
#   'Feature_497',
#   'Feature_498',
#   'Feature_516',
#   'Feature_518',
#   'Feature_520',
#   'Feature_536',
#   'Feature_538',
#   'Feature_560',
#   'Feature_563',
#   'Feature_572',
#   'Feature_581',
#   'Feature_587',
#   'Feature_631',
#   'Feature_632',
#   'Feature_640',
#   'Feature_641',
#   'Feature_651',
#   'Feature_652',
#   'Feature_653',
#   'Feature_654',
#   'Feature_659',
#   'Feature_662',
#   'Feature_683',
#   'Feature_708',
#   'Feature_709',
#   'Feature_711',
#   'Feature_715'],
#  ['Feature_1',
#   'Feature_2',
#   'Feature_3',
#   'Feature_4',
#   'Feature_5',
#   'Feature_24',
#   'Feature_89',
#   'Feature_101',
#   'Feature_108',
#   'Feature_113',
#   'Feature_139',
#   'Feature_175',
#   'Feature_179',
#   'Feature_187',
#   'Feature_211',
#   'Feature_215',
#   'Feature_216',
#   'Feature_237',
#   'Feature_240',
#   'Feature_241',
#   'Feature_252',
#   'Feature_261',
#   'Feature_347',
#   'Feature_350',
#   'Feature_414',
#   'Feature_415',
#   'Feature_417',
#   'Feature_445',
#   'Feature_474',
#   'Feature_554',
#   'Feature_560',
#   'Feature_563',
#   'Feature_578',
#   'Feature_586',
#   'Feature_622',
#   'Feature_637',
#   'Feature_652',
#   'Feature_654'],
#  ['Feature_1',
#   'Feature_4',
#   'Feature_25',
#   'Feature_111',
#   'Feature_172',
#   'Feature_179',
#   'Feature_215',
#   'Feature_216',
#   'Feature_229',
#   'Feature_232',
#   'Feature_235',
#   'Feature_237',
#   'Feature_321',
#   'Feature_341',
#   'Feature_412',
#   'Feature_417',
#   'Feature_474',
#   'Feature_518',
#   'Feature_578',
#   'Feature_587',
#   'Feature_593',
#   'Feature_631',
#   'Feature_641',
#   'Feature_657',
#   'Feature_711'],
#  ['Feature_1',
#   'Feature_2',
#   'Feature_3',
#   'Feature_4',
#   'Feature_5',
#   'Feature_6',
#   'Feature_7',
#   'Feature_15',
#   'Feature_24',
#   'Feature_36',
#   'Feature_80',
#   'Feature_84',
#   'Feature_88',
#   'Feature_89',
#   'Feature_97',
#   'Feature_104',
#   'Feature_107',
#   'Feature_113',
#   'Feature_145',
#   'Feature_157',
#   'Feature_166',
#   'Feature_175',
#   'Feature_178',
#   'Feature_179',
#   'Feature_180',
#   'Feature_183',
#   'Feature_188',
#   'Feature_189',
#   'Feature_212',
#   'Feature_215',
#   'Feature_216',
#   'Feature_219',
#   'Feature_226',
#   'Feature_229',
#   'Feature_230',
#   'Feature_235',
#   'Feature_237',
#   'Feature_241',
#   'Feature_252',
#   'Feature_325',
#   'Feature_335',
#   'Feature_338',
#   'Feature_340',
#   'Feature_350',
#   'Feature_354',
#   'Feature_367',
#   'Feature_391',
#   'Feature_394',
#   'Feature_404',
#   'Feature_417',
#   'Feature_418',
#   'Feature_422',
#   'Feature_426',
#   'Feature_446',
#   'Feature_474',
#   'Feature_514',
#   'Feature_521',
#   'Feature_556',
#   'Feature_560',
#   'Feature_587',
#   'Feature_613',
#   'Feature_631',
#   'Feature_640',
#   'Feature_641',
#   'Feature_651',
#   'Feature_652',
#   'Feature_653',
#   'Feature_654',
#   'Feature_656',
#   'Feature_709',
#   'Feature_711',
#   'Feature_715'],
#  ['Feature_1',
#   'Feature_2',
#   'Feature_5',
#   'Feature_14',
#   'Feature_88',
#   'Feature_89',
#   'Feature_113',
#   'Feature_174',
#   'Feature_181',
#   'Feature_230',
#   'Feature_237',
#   'Feature_241',
#   'Feature_338',
#   'Feature_391',
#   'Feature_394',
#   'Feature_409',
#   'Feature_520',
#   'Feature_641',
#   'Feature_709',
#   'Feature_715']]

In [19]:
# LGBM
top_features_list = [['Feature_1',
  'Feature_2',
  'Feature_3',
  'Feature_6',
  'Feature_24',
  'Feature_46',
  'Feature_47',
  'Feature_86',
  'Feature_88',
  'Feature_113',
  'Feature_148',
  'Feature_157',
  'Feature_173',
  'Feature_178',
  'Feature_179',
  'Feature_180',
  'Feature_183',
  'Feature_188',
  'Feature_215',
  'Feature_216',
  'Feature_235',
  'Feature_237',
  'Feature_241',
  'Feature_282',
  'Feature_325',
  'Feature_404',
  'Feature_410',
  'Feature_417',
  'Feature_425',
  'Feature_472',
  'Feature_474',
  'Feature_478',
  'Feature_518',
  'Feature_521',
  'Feature_572',
  'Feature_582',
  'Feature_584',
  'Feature_587',
  'Feature_598',
  'Feature_604',
  'Feature_631',
  'Feature_640',
  'Feature_651',
  'Feature_652',
  'Feature_653',
  'Feature_654',
  'Feature_709',
  'Feature_711',
  'Feature_715'],
 ['Feature_1',
  'Feature_2',
  'Feature_3',
  'Feature_4',
  'Feature_15',
  'Feature_19',
  'Feature_68',
  'Feature_89',
  'Feature_101',
  'Feature_113',
  'Feature_157',
  'Feature_166',
  'Feature_176',
  'Feature_177',
  'Feature_178',
  'Feature_179',
  'Feature_180',
  'Feature_211',
  'Feature_215',
  'Feature_216',
  'Feature_229',
  'Feature_230',
  'Feature_235',
  'Feature_237',
  'Feature_241',
  'Feature_325',
  'Feature_326',
  'Feature_416',
  'Feature_417',
  'Feature_420',
  'Feature_472',
  'Feature_478',
  'Feature_489',
  'Feature_520',
  'Feature_542',
  'Feature_560',
  'Feature_578',
  'Feature_586',
  'Feature_587',
  'Feature_604',
  'Feature_613',
  'Feature_651',
  'Feature_652',
  'Feature_653',
  'Feature_654',
  'Feature_662',
  'Feature_683',
  'Feature_711',
  'Feature_715'],
 ['Feature_1',
  'Feature_2',
  'Feature_3',
  'Feature_4',
  'Feature_6',
  'Feature_15',
  'Feature_24',
  'Feature_32',
  'Feature_40',
  'Feature_44',
  'Feature_58',
  'Feature_88',
  'Feature_89',
  'Feature_112',
  'Feature_113',
  'Feature_148',
  'Feature_157',
  'Feature_166',
  'Feature_167',
  'Feature_173',
  'Feature_175',
  'Feature_176',
  'Feature_177',
  'Feature_178',
  'Feature_179',
  'Feature_180',
  'Feature_182',
  'Feature_211',
  'Feature_215',
  'Feature_216',
  'Feature_229',
  'Feature_230',
  'Feature_235',
  'Feature_237',
  'Feature_240',
  'Feature_241',
  'Feature_251',
  'Feature_261',
  'Feature_277',
  'Feature_281',
  'Feature_301',
  'Feature_325',
  'Feature_326',
  'Feature_335',
  'Feature_338',
  'Feature_346',
  'Feature_349',
  'Feature_350',
  'Feature_356',
  'Feature_360',
  'Feature_394',
  'Feature_403',
  'Feature_404',
  'Feature_412',
  'Feature_413',
  'Feature_414',
  'Feature_415',
  'Feature_416',
  'Feature_417',
  'Feature_422',
  'Feature_446',
  'Feature_472',
  'Feature_480',
  'Feature_489',
  'Feature_498',
  'Feature_514',
  'Feature_516',
  'Feature_518',
  'Feature_538',
  'Feature_583',
  'Feature_584',
  'Feature_585',
  'Feature_586',
  'Feature_593',
  'Feature_597',
  'Feature_598',
  'Feature_622',
  'Feature_628',
  'Feature_631',
  'Feature_647',
  'Feature_649',
  'Feature_651',
  'Feature_652',
  'Feature_654',
  'Feature_682',
  'Feature_709',
  'Feature_711'],
 ['Feature_1',
  'Feature_3',
  'Feature_4',
  'Feature_5',
  'Feature_6',
  'Feature_14',
  'Feature_24',
  'Feature_83',
  'Feature_84',
  'Feature_88',
  'Feature_107',
  'Feature_111',
  'Feature_113',
  'Feature_157',
  'Feature_167',
  'Feature_174',
  'Feature_177',
  'Feature_178',
  'Feature_179',
  'Feature_211',
  'Feature_212',
  'Feature_215',
  'Feature_216',
  'Feature_220',
  'Feature_229',
  'Feature_230',
  'Feature_235',
  'Feature_237',
  'Feature_241',
  'Feature_261',
  'Feature_347',
  'Feature_348',
  'Feature_349',
  'Feature_385',
  'Feature_394',
  'Feature_412',
  'Feature_416',
  'Feature_418',
  'Feature_422',
  'Feature_433',
  'Feature_472',
  'Feature_478',
  'Feature_497',
  'Feature_560',
  'Feature_562',
  'Feature_563',
  'Feature_581',
  'Feature_582',
  'Feature_584',
  'Feature_587',
  'Feature_631',
  'Feature_640',
  'Feature_650',
  'Feature_653',
  'Feature_654',
  'Feature_659',
  'Feature_711',
  'Feature_715'],
 ['Feature_1',
  'Feature_2',
  'Feature_3',
  'Feature_98',
  'Feature_112',
  'Feature_113',
  'Feature_183',
  'Feature_188',
  'Feature_211',
  'Feature_215',
  'Feature_216',
  'Feature_235',
  'Feature_237',
  'Feature_241',
  'Feature_417',
  'Feature_518',
  'Feature_538',
  'Feature_563',
  'Feature_652',
  'Feature_653',
  'Feature_715'],
 ['Feature_1',
  'Feature_4',
  'Feature_6',
  'Feature_101',
  'Feature_107',
  'Feature_110',
  'Feature_112',
  'Feature_113',
  'Feature_157',
  'Feature_166',
  'Feature_176',
  'Feature_179',
  'Feature_187',
  'Feature_212',
  'Feature_215',
  'Feature_216',
  'Feature_229',
  'Feature_235',
  'Feature_237',
  'Feature_240',
  'Feature_241',
  'Feature_261',
  'Feature_269',
  'Feature_338',
  'Feature_350',
  'Feature_379',
  'Feature_394',
  'Feature_410',
  'Feature_413',
  'Feature_420',
  'Feature_425',
  'Feature_446',
  'Feature_474',
  'Feature_480',
  'Feature_517',
  'Feature_538',
  'Feature_554',
  'Feature_563',
  'Feature_572',
  'Feature_586',
  'Feature_587',
  'Feature_631',
  'Feature_646',
  'Feature_650',
  'Feature_654'],
 ['Feature_1',
  'Feature_2',
  'Feature_3',
  'Feature_4',
  'Feature_6',
  'Feature_15',
  'Feature_24',
  'Feature_82',
  'Feature_84',
  'Feature_89',
  'Feature_113',
  'Feature_179',
  'Feature_180',
  'Feature_183',
  'Feature_215',
  'Feature_216',
  'Feature_220',
  'Feature_229',
  'Feature_235',
  'Feature_236',
  'Feature_237',
  'Feature_241',
  'Feature_280',
  'Feature_321',
  'Feature_326',
  'Feature_341',
  'Feature_350',
  'Feature_360',
  'Feature_385',
  'Feature_409',
  'Feature_412',
  'Feature_416',
  'Feature_417',
  'Feature_426',
  'Feature_445',
  'Feature_446',
  'Feature_474',
  'Feature_497',
  'Feature_498',
  'Feature_516',
  'Feature_518',
  'Feature_558',
  'Feature_562',
  'Feature_563',
  'Feature_575',
  'Feature_593',
  'Feature_604',
  'Feature_631',
  'Feature_646',
  'Feature_654',
  'Feature_657',
  'Feature_658',
  'Feature_683',
  'Feature_711'],
 ['Feature_1',
  'Feature_2',
  'Feature_3',
  'Feature_4',
  'Feature_5',
  'Feature_14',
  'Feature_15',
  'Feature_24',
  'Feature_42',
  'Feature_70',
  'Feature_88',
  'Feature_89',
  'Feature_104',
  'Feature_107',
  'Feature_108',
  'Feature_112',
  'Feature_113',
  'Feature_123',
  'Feature_136',
  'Feature_148',
  'Feature_157',
  'Feature_163',
  'Feature_166',
  'Feature_170',
  'Feature_173',
  'Feature_175',
  'Feature_176',
  'Feature_177',
  'Feature_178',
  'Feature_179',
  'Feature_180',
  'Feature_185',
  'Feature_189',
  'Feature_201',
  'Feature_212',
  'Feature_215',
  'Feature_216',
  'Feature_217',
  'Feature_219',
  'Feature_220',
  'Feature_229',
  'Feature_230',
  'Feature_232',
  'Feature_233',
  'Feature_235',
  'Feature_237',
  'Feature_240',
  'Feature_241',
  'Feature_325',
  'Feature_335',
  'Feature_344',
  'Feature_346',
  'Feature_348',
  'Feature_349',
  'Feature_350',
  'Feature_360',
  'Feature_362',
  'Feature_394',
  'Feature_410',
  'Feature_414',
  'Feature_425',
  'Feature_426',
  'Feature_474',
  'Feature_489',
  'Feature_498',
  'Feature_499',
  'Feature_506',
  'Feature_514',
  'Feature_556',
  'Feature_560',
  'Feature_575',
  'Feature_582',
  'Feature_584',
  'Feature_586',
  'Feature_587',
  'Feature_613',
  'Feature_631',
  'Feature_640',
  'Feature_641',
  'Feature_643',
  'Feature_647',
  'Feature_651',
  'Feature_652',
  'Feature_653',
  'Feature_654',
  'Feature_656',
  'Feature_657',
  'Feature_659',
  'Feature_662',
  'Feature_667',
  'Feature_683',
  'Feature_709',
  'Feature_711'],
 ['Feature_1',
  'Feature_2',
  'Feature_4',
  'Feature_46',
  'Feature_81',
  'Feature_111',
  'Feature_113',
  'Feature_157',
  'Feature_179',
  'Feature_212',
  'Feature_236',
  'Feature_237',
  'Feature_241',
  'Feature_326',
  'Feature_394',
  'Feature_498',
  'Feature_587',
  'Feature_631',
  'Feature_654',
  'Feature_715']]

In [20]:
for i, l in enumerate(top_features_list):
    print(f'Fold {i+1}: {len(l)} features')

Fold 1: 49 features
Fold 2: 49 features
Fold 3: 87 features
Fold 4: 58 features
Fold 5: 21 features
Fold 6: 45 features
Fold 7: 54 features
Fold 8: 93 features
Fold 9: 20 features


In [21]:
intersection = set(top_features_list[0])
for lst in top_features_list[1:]:
    intersection.intersection_update(lst)
intersection = list(intersection)
intersection.sort()
intersection

['Feature_1', 'Feature_113', 'Feature_237', 'Feature_241']

In [22]:
def calculate_stability_score(y_true, pred, class1_proportion):
    """
    Function to calculate stability using true values, probabilities and profitable trades proportion
    """
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    if fp == 0:
        return np.inf if tp > 0 else 0.0
    if class1_proportion == 0:
        return 0.0
    stability = (tp / fp) / class1_proportion
    return stability

###################################################################################################################

def evaluate_model(y_true, y_prob, model_name, class1_proportion_val, thresholds=None):
    """
    Simplified evaluation function with built-in optimal threshold finding,
    with a filter to avoid selecting unrealistic 100% precision/recall.
    Input: True values, probability predictions, profitable trades proportion, explicit thresholds (optional)
    Returns metrics dictionary.
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 100) # a range of 100 threshold option
        
    thresholds = [t for t in thresholds if 0.001 < t < 0.999] # sanity check
    results = []
    
    # Find optimal threshold
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        
        # Get confusion matrix components
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        
        # Calculate evaluation metrics
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        stability = calculate_stability_score(y_true, y_pred, class1_proportion_val)
        
        # Saving metrics and confusion matrics components for later filtering
        results.append({
            'threshold': threshold,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'stability': stability,
            'tn': tn,
            'fp': fp,
            'fn': fn,
            'tp': tp,
            'score': (tp * 800) - (fp * 200) - (fn * 200),
            'score_1': (tp * 800) - (fp * 200),
        })

    if not results:
        threshold = 0.5
        y_pred = (y_prob >= threshold).astype(int)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        stability = calculate_stability_score(y_true, y_pred, class1_proportion_val)
        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel()
        score = (tp * 800) - (fp * 200) - (fn * 200)
        score_1 = (tp * 800) - (fp * 200)
    else:
        results_df = pd.DataFrame(results)

        # This to consider options where false positive, false negative and true negative > 10 instances otherwise it's misleading.
        misleading_options = (results_df['fp'] < 10) | (results_df['fn'] < 10) | (results_df['tn'] < 10)
        
        filtered_df = results_df[~(misleading_options)].copy()

        # Fallback if filtering removes all candidates (if all predictions are extreme), use original results if all are deemed extreme
        if filtered_df.empty:
            print(f"Warning: All candidates filtered out for {model_name}. Reverting to original selection without extreme filtering.")
            filtered_df = results_df.copy()
        else:
            print(f"Extreme filtering is done, original results {results_df.shape[0]} row, filtered {filtered_df.shape[0]}")

        # Find best threshold from the filtered df
        # If more than instance have the same max score_1, we take the higher f1
        max_score_1 = filtered_df['score_1'].max()
        best_candidates = filtered_df[filtered_df['score_1'] == max_score_1]
        best_row = best_candidates.loc[best_candidates['precision'].idxmax()]
        
        threshold = best_row['threshold']
        precision = best_row['precision']
        recall = best_row['recall']
        f1 = best_row['f1']
        stability = best_row['stability']
        score = best_row['score']
        score_1 = best_row['score_1']
        
        # Re-calculate confusion matrix for the final selected optimal threshold
        y_pred = (y_prob >= threshold).astype(int)
        cm = confusion_matrix(y_true, y_pred)
        
    return {
        'model': model_name,
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'stability': stability,
        'confusion_matrix': cm,
        'score': score,
        'score_1': score_1
    }

###################################################################################################################

n_features = 715
def preprocess_data(train, valid=None, n_regimes=3):

    """
    This function does the following:
    1. SHAP-Based Feature Selection to select top 50 most important features for market regime identification.
    2. Apply un-supervised learning using GMM to predict regime.
    3. Apply multiplicative interaction on the selected top x% features (excluding regime)
    Returns 
        a. Engineered features and labels for training and validation set (where applicable)
        b. The fitted objects used for scaling features and regime predictions
        c. Top features as list
        Note: b & c will be used for applying on test data.
    """

    # Assigning features and labels variables
    X_train, y_train = train.drop(columns=['Time','class']), train['class']
    X_valid, y_valid = valid.drop(columns=['Time','class']), valid['class']
    
    # Class weight = negative class n / negative class n
    class_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

    # Proxy for the intesection of top feature
    gmm_input_features = intersection

    # Step 2
    gmm_data_tr = X_train[gmm_input_features]
    gmm_data_va = X_valid[gmm_input_features]
    # Scale features for GMM
    scaler_gmm = StandardScaler()
    gmm_data_tr_scaled = scaler_gmm.fit_transform(gmm_data_tr)
    gmm_data_va_scaled = scaler_gmm.transform(gmm_data_va)
    # Train GMM and predict regimes
    try:
        gmm = GaussianMixture(n_components=n_regimes, random_state=seed, covariance_type='full', max_iter=200, tol=1e-3)
        gmm.fit(gmm_data_tr_scaled)
        regime_labels_tr = gmm.predict(gmm_data_tr_scaled)
        regime_labels_va = gmm.predict(gmm_data_va_scaled)
    except ValueError as e: # sometimes GMM fails so we assign regime 0
        print(f"Warning: GMM failed with error: {e}. Assigning default regime 0.")
        regime_labels_tr = np.zeros(len(X_train), dtype=int)
        regime_labels_va = np.zeros(len(X_valid), dtype=int)
    # Add regime labels
    X_train['Regime'] = regime_labels_tr
    X_valid['Regime'] = regime_labels_va
    # Convert regime column to 'category'
    X_train['Regime'] = X_train['Regime'].astype('category')
    X_valid['Regime'] = X_valid['Regime'].astype('category')

    # Step 3
    X_train_interacted = X_train.copy()
    X_valid_interacted = X_valid.copy()
    features_for_interaction = intersection

    for i in range(len(features_for_interaction)):
        for j in range(i + 1, len(features_for_interaction)):
            feature1 = features_for_interaction[i]
            feature2 = features_for_interaction[j]
            X_train_interacted[f'{feature1}_x_{feature2}'] = X_train_interacted[feature1] * X_train_interacted[feature2]
            X_valid_interacted[f'{feature1}_x_{feature2}'] = X_valid_interacted[feature1] * X_valid_interacted[feature2]
    
    x_features = [col for col in X_train_interacted.columns if '_x_' in col]
    all_features = intersection + x_features + ['Regime']

    return X_train_interacted[all_features], y_train, X_valid_interacted[all_features], y_valid, scaler_gmm, gmm, intersection


In [9]:
# ###################################################################################################################

# # For hyperparameters tuning we have 3 param_distribution for search per model (Tree-based only)
# params_search_cat = {'depth': [4, 6, 8, 10],'learning_rate': [0.1, 0.05, 0.01, 0.005],'l2_leaf_reg': [1, 3, 5, 7, 9, 11],'bagging_temperature': [0, 0.5, 1, 3, 5],'subsample': [0.5, 0.7, 0.9, 1],'random_strength': [1e-9, 1e-6, 1e-3, 1e-1],'border_count': [32, 64, 128, 254]}
# params_search_xgb = {'learning_rate': [0.1, 0.05, 0.01, 0.005],'max_depth': [3, 6, 10],'min_child_weight': [1, 3, 5],'subsample': [0.6, 0.8, 1.0],'colsample_bytree': [0.6, 0.8, 1.0],'gamma': [0, 0.1, 0.5],'reg_alpha': [0, 0.1, 1],'reg_lambda': [1, 5, 10]}
# params_search_lgb = {'learning_rate': [0.1, 0.05, 0.01, 0.005],'max_depth': [3, 5, 7, 10, -1],'num_leaves': [15, 31, 63, 127],'min_child_samples': [10, 20, 30, 50],'subsample': [0.6, 0.8, 1.0],'colsample_bytree': [0.6, 0.8, 1.0],'reg_alpha': [0, 0.01, 0.1, 1],'reg_lambda': [0, 0.01, 0.1, 1],}

# ###################################################################################################################


In [23]:
def train_base_model(X_train, y_train, X_valid=None, y_valid=None, model_name='CatBoost'):

    """
    This function:
    1. Use param_search_xxx to tune hyperparameter the base model before fitting
    2. Works according to input model_name argument ['CatBoost','XGBoost','LogReg','LGBM']
    3. Predict probabilities for the selected model
    Retruns:
        a. Probability predictions
        b. The model itself as object
        c. Feature scaler as object (only for LogReg)
        Note: b & c will be used for applying on test data.
    """
    
    class_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
    feature_scaler = None
    print(f'Hyperparameter tuning, base model: {model_name}, {datetime.now()}')
    if model_name == 'CatBoost':
        model = cat.CatBoostClassifier(random_state=seed, scale_pos_weight=class_weight, loss_function='Logloss',  
                                       eval_metric='F1', early_stopping_rounds=50, iterations=1000, verbose=0)
        def cat_features_regime(cols):
            # This function is dedicated for catboost model.fit() to accomodate for Regime column as categorical feature
            if 'Regime' in cols:
                return ['Regime']
            return None
        # search = RandomizedSearchCV(estimator=model, param_distributions=params_search_cat, n_iter=10, cv=TimeSeriesSplit(n_splits=3), random_state=seed, scoring='f1', verbose=0, n_jobs=-1)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], cat_features=cat_features_regime(list(X_train.columns)))
    elif model_name == 'XGBoost':
        model = xgb.XGBClassifier(objective='binary:logistic', scale_pos_weight=class_weight, random_state=seed, n_estimators=1000, early_stopping_rounds=10, enable_categorical=True, tree_method='hist', verbosity=0)
        # search = RandomizedSearchCV(estimator=model, param_distributions=params_search_xgb, n_iter=10, cv=TimeSeriesSplit(n_splits=3), random_state=seed, scoring='f1', verbose=0, n_jobs=-1)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    elif model_name == 'LogReg':
        feature_scaler = StandardScaler()
        x_tr_scaled = feature_scaler.fit_transform(X_train)
        model = LogisticRegression(max_iter=1000, random_state=seed, solver='liblinear', class_weight={0: 1, 1: class_weight}, C=0.01, penalty='l1')
        model.fit(x_tr_scaled, y_train)
    elif model_name == 'LGBM':
        model = lgb.LGBMClassifier(class_weight='balanced', objective='binary', random_state=seed, learning_rate= 0.01, verbosity=-1, n_estimators=1000)
        # search = RandomizedSearchCV(estimator=model, param_distributions=params_search_lgb, n_iter=10, cv=TimeSeriesSplit(n_splits=3), random_state=seed, scoring='f1', verbose=0, n_jobs=-1)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], eval_metric="binary_logloss", callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)])

    # if model_name != 'LogReg':
    #     model = search.best_estimator_

    print(f'Base model {model_name} is completed, {datetime.now()}')
    try:
        prob = model.predict_proba(X_valid)[:, 1]
    except Exception as e:
        print(f"{model} failed: {e}. Filling with zeros.")
        prob = np.zeros(len(y_valid))

    return prob, model, feature_scaler


## Train Validate Section

In [24]:
###################################################################################################################
# Training base models on full data (or partial)
###################################################################################################################

# Input training start date and end date
# Validation start date: will be automatically set to the next day after training end date
# Validation end date: will be automatically set to the last row in 2023 data
# Input Regime count for GMM to generate

input_train_start = '2021-01-01'
input_train_end   = '2022-12-31'
regimes = 3


tr_start_date = datetime.strptime(input_train_start, "%Y-%m-%d")
tr_end_date = datetime.strptime(input_train_end, "%Y-%m-%d")
va_start_date = tr_end_date + timedelta(days=1)
va_end_date = train_val_df["Time"].max()
training_data = train_val_df[(train_val_df["Time"] >= tr_start_date) & (train_val_df["Time"] <= tr_end_date)]
validation_data = train_val_df[(train_val_df["Time"] >= va_start_date) & (train_val_df["Time"] <= va_end_date)]
print(f"Training Period: {tr_start_date.strftime('%Y-%m-%d')} - {tr_end_date.strftime('%Y-%m-%d')} | Validation Period: {va_start_date.strftime('%Y-%m-%d')} - {va_end_date.strftime('%Y-%m-%d')}.")

# Apply data preprocessing
X_train_interacted, y_train, X_valid_interacted, y_valid, scaler_gmm, gmm, intersected_features = preprocess_data(training_data, validation_data, n_regimes=regimes)

# Collect predictions from base model
cat_prob, cat_model, _ = train_base_model(X_train_interacted, y_train, X_valid_interacted, y_valid, model_name='CatBoost')
xgb_prob, xgb_model, _ = train_base_model(X_train_interacted, y_train, X_valid_interacted, y_valid, model_name='XGBoost')
lr_prob, lr_model, feature_scaler = train_base_model(X_train_interacted, y_train, X_valid_interacted, y_valid, model_name='LogReg')
lgb_prob, lgb_model, _ = train_base_model(X_train_interacted, y_train, X_valid_interacted, y_valid, model_name='LGBM')

###################################### Check validation metrics #########################################

class1_proportion = y_valid.value_counts(normalize=True)[1]

cat_metrics = evaluate_model(y_valid, cat_prob, "CatBoost", class1_proportion)
xgb_metrics = evaluate_model(y_valid, xgb_prob, "XGBoost", class1_proportion)
lr_metrics = evaluate_model(y_valid, lr_prob, "LogReg", class1_proportion)
lgb_metrics = evaluate_model(y_valid, lgb_prob, "LGBM", class1_proportion)

full_data_metrics = pd.DataFrame([cat_metrics, xgb_metrics, lr_metrics, lgb_metrics])
print(full_data_metrics.round(3).to_string(max_colwidth=50), '\n')

Training Period: 2021-01-01 - 2022-12-31 | Validation Period: 2023-01-01 - 2023-12-29.
Hyperparameter tuning, base model: CatBoost, 2025-08-11 13:03:44.405892
Base model CatBoost is completed, 2025-08-11 13:03:46.330443
Hyperparameter tuning, base model: XGBoost, 2025-08-11 13:03:46.332490
Base model XGBoost is completed, 2025-08-11 13:03:46.365480
Hyperparameter tuning, base model: LogReg, 2025-08-11 13:03:46.369484
Base model LogReg is completed, 2025-08-11 13:03:46.380477
Hyperparameter tuning, base model: LGBM, 2025-08-11 13:03:46.383479
Base model LGBM is completed, 2025-08-11 13:03:46.429850
Extreme filtering is done, original results 100 row, filtered 8
Extreme filtering is done, original results 100 row, filtered 58
Extreme filtering is done, original results 100 row, filtered 15
      model  threshold  precision  recall     f1  stability           confusion_matrix    score  score_1
0  CatBoost      0.446      0.235   0.979  0.379      1.334    [[83, 1699], [11, 521]]  74800.0 

In [12]:
###################################################################################################################
# Training the meta learner
###################################################################################################################

meta_X_train = pd.DataFrame(
    {
        "cat_prob": cat_prob,
        "xgb_prob": xgb_prob,
        "logreg_prob": lr_prob,
        "lgbm_prob": lgb_prob,
    }
)
meta_y_train = y_valid

final_meta_learner = LogisticRegression(max_iter=1000)
final_meta_learner.fit(meta_X_train, meta_y_train)

###################################################################################################################

LogisticRegression(max_iter=1000)

In [25]:
def preprocess_test_data(test_df, start_date, end_date, scaler_gmm, gmm, intersected_features):

    """
    This function 
    1. We input the testing dataset, start data and end date of the interval in question.
    2. It applies the same preprocessing we made on the training data, using the output objects from preprocess_data function
    Returns engineered features and labels for the interval period from testing data
    """

    test_df = test_df[(test_df["Time"] >= start_date) & (test_df["Time"] <= end_date)]

    x_test = test_df.drop(columns=["Time", "class"])
    y_test = test_df["class"]

    scaled_x_test_array = scaler_gmm.transform(x_test[scaler_gmm.feature_names_in_]) # Apply the scaling on features 
    scaled_x_test_df = pd.DataFrame(data=scaled_x_test_array, columns=x_test[scaler_gmm.feature_names_in_].columns) # Convert scaled features to df
    test_regime_labels = gmm.predict(scaled_x_test_df) # Predict regime labels
    x_test_selected = x_test[intersected_features] # Align top features with training set
    # Applying features interaction on the selected ones
    x_test_interacted = x_test_selected.copy()
    features_for_interaction = intersected_features
    for i in range(len(features_for_interaction)):
            for j in range(i + 1, len(features_for_interaction)):
                feature1 = features_for_interaction[i]
                feature2 = features_for_interaction[j]
                x_test_interacted[f'{feature1}_x_{feature2}'] = x_test_interacted[feature1] * x_test_interacted[feature2]
    x_test_interacted['Regime'] = test_regime_labels # add regime labels
    x_test_interacted['Regime'] = x_test_interacted['Regime'].astype('category')
    # Check the features alignment between training & testing sets
    print(X_train_interacted.columns)
    assert list(X_train_interacted.columns) == list(x_test_interacted.columns)

    return x_test_interacted, y_test

## 2024 Test Data

In [14]:
###################################################################################################################
# Test results 2024
###################################################################################################################
# test_df = pd.read_csv('SE5.csv')
# test_df[test_df['Time']>='2024-01-01']

# Preprocess testing data
x_test_interacted, y_test = preprocess_test_data(test_df=test_df, start_date='2024-01-01', end_date='2024-12-31', scaler_gmm=scaler_gmm, gmm=gmm, intersected_features=intersected_features)

# Collect predictions from the fitted base models
cat_pred_test = cat_model.predict_proba(x_test_interacted)[:, 1]
xgb_pred_test = xgb_model.predict_proba(x_test_interacted)[:, 1]
lr_pred_test = lr_model.predict_proba(feature_scaler.transform(x_test_interacted))[:, 1]
lgbm_pred_test = lgb_model.predict_proba(x_test_interacted)[:, 1]

# Populate the predictions (inputs for testing on the meta learner)
meta_X_test = pd.DataFrame({"cat_prob": cat_pred_test,"xgb_prob": xgb_pred_test,"logreg_prob": lr_pred_test,"lgbm_prob": lgbm_pred_test})
meta_y_test = y_test

# Predict probability using the meta learner
meta_pred_prob = final_meta_learner.predict_proba(meta_X_test)[:, 1]

# Evaluate and print results of testing data
class1_proportion_test = y_test.value_counts(normalize=True)[1]
test_metrics = evaluate_model(y_test, meta_pred_prob, 'Stack Ensemble', class1_proportion_test)

# Saving evaluation metrics
optimal_thresh = test_metrics['threshold']

# Populate final stack ensemble predictions based on probabilities
final_pred = (meta_pred_prob > optimal_thresh).astype(int)

final_results = pd.DataFrame([test_metrics])

# Printing metrics
print(final_results.round(3).to_string(max_colwidth=50))

cat_metrics_test = evaluate_model(y_test, cat_pred_test, 'CatBoost', class1_proportion_test)
xgb_metrics_test = evaluate_model(y_test, xgb_pred_test, 'XGBoost', class1_proportion_test)
lr_metrics_test = evaluate_model(y_test, lr_pred_test, 'LogReg', class1_proportion_test)
lgbm_metrics_test = evaluate_model(y_test, lgbm_pred_test, 'LGBM', class1_proportion_test)

test_2024_results_base_models = pd.DataFrame([cat_metrics_test,xgb_metrics_test,lr_metrics_test,lgbm_metrics_test])
print(test_2024_results_base_models.round(3).to_string(max_colwidth=50))

Time = test_df['Time']
binary_predictions_2024_test = pd.DataFrame(
    {
        "cat_pred": (meta_X_test["cat_prob"] > test_2024_results_base_models['threshold'][0]).astype(int),
        "xgb_pred": (meta_X_test["xgb_prob"] > test_2024_results_base_models['threshold'][1]).astype(int),
        "logreg_pred": (meta_X_test["logreg_prob"] > test_2024_results_base_models['threshold'][2]).astype(int),
        "lgbm_pred": (meta_X_test["lgbm_prob"] > test_2024_results_base_models['threshold'][3]).astype(int),
        "Stack Ensemble": pd.Series(final_pred, name='Stack Ensemble'),
        "Time": Time.values,
        'True Class': y_test.values
    }
)
binary_predictions_2024_test.to_csv("binary_predictions_2024_test.csv", index=False)
print('Binary prediction file generated')

Index(['Feature_1', 'Feature_113', 'Feature_237', 'Feature_241',
       'Feature_1_x_Feature_113', 'Feature_1_x_Feature_237',
       'Feature_1_x_Feature_241', 'Feature_113_x_Feature_237',
       'Feature_113_x_Feature_241', 'Feature_237_x_Feature_241', 'Regime'],
      dtype='object')
Extreme filtering is done, original results 100 row, filtered 8
            model  threshold  precision  recall     f1  stability          confusion_matrix    score  score_1
0  Stack Ensemble      0.198      0.264   0.176  0.211      1.944  [[1677, 209], [352, 75]] -52200.0  18200.0
Extreme filtering is done, original results 100 row, filtered 8
Extreme filtering is done, original results 100 row, filtered 60
Extreme filtering is done, original results 100 row, filtered 26
Extreme filtering is done, original results 100 row, filtered 15
      model  threshold  precision  recall     f1  stability           confusion_matrix    score  score_1
0  CatBoost      0.505      0.187   0.415  0.257      1.242  [[11

## 2025 Test Data

In [15]:
###################################################################################################################
# Test results 2025
###################################################################################################################

test_start_date = forward_test_df['Time'].min() 

test_end_end = forward_test_df['Time'].max()

# Preprocess testing data
x_test_interacted_25, y_test_25 = preprocess_test_data(test_df=forward_test_df, start_date=test_start_date, end_date=test_end_end, scaler_gmm=scaler_gmm, gmm=gmm, intersected_features=intersected_features)

# Collect predictions from the fitted base models
cat_pred_test_25 = cat_model.predict_proba(x_test_interacted_25)[:, 1]
xgb_pred_test_25 = xgb_model.predict_proba(x_test_interacted_25)[:, 1]
lr_pred_test_25 = lr_model.predict_proba(feature_scaler.transform(x_test_interacted_25))[:, 1]
lgbm_pred_test_25 = lgb_model.predict_proba(x_test_interacted_25)[:, 1]

# Populate the predictions (inputs for testing on the meta learner)
meta_X_test_25 = pd.DataFrame({"cat_prob": cat_pred_test_25,"xgb_prob": xgb_pred_test_25,"logreg_prob": lr_pred_test_25,"lgbm_prob": lgbm_pred_test_25})
meta_y_test_25 = y_test_25

# Predict probability using the meta learner
meta_pred_prob_25 = final_meta_learner.predict_proba(meta_X_test_25)[:, 1]

# Evaluate and print results of testing data
class1_proportion_test_25 = y_test_25.value_counts(normalize=True)[1]
test_metrics_25 = evaluate_model(y_test_25, meta_pred_prob_25, 'Stack Ensemble', class1_proportion_test_25)

# Saving evaluation metrics
optimal_thresh_25 = test_metrics_25['threshold']

# Populate final stack ensemble predictions based on probabilities
final_pred_25 = (meta_pred_prob_25 > optimal_thresh_25).astype(int)

final_results_25 = pd.DataFrame([test_metrics_25])

# Printing metrics
print(final_results_25.round(3).to_string(max_colwidth=50))
cat_metrics_test_25 = evaluate_model(y_test_25, cat_pred_test_25, 'CatBoost', class1_proportion_test_25)
xgb_metrics_test_25 = evaluate_model(y_test_25, xgb_pred_test_25, 'XGBoost', class1_proportion_test_25)
lr_metrics_test_25 = evaluate_model(y_test_25, lr_pred_test_25, 'LogReg', class1_proportion_test_25)
lgbm_metrics_test_25 = evaluate_model(y_test_25, lgbm_pred_test_25, 'LGBM', class1_proportion_test_25)

test_2025_results_base_models = pd.DataFrame([cat_metrics_test_25,xgb_metrics_test_25,lr_metrics_test_25,lgbm_metrics_test_25])
print(test_2025_results_base_models.round(3).to_string(max_colwidth=50))
CatBoost_thresh = full_data_metrics.loc[full_data_metrics['model']=='CatBoost', 'threshold'].values[0]
XGBoost_thresh = full_data_metrics.loc[full_data_metrics['model']=='XGBoost', 'threshold'].values[0]
LogReg_thresh = full_data_metrics.loc[full_data_metrics['model']=='LogReg', 'threshold'].values[0]
LGBM_thresh = full_data_metrics.loc[full_data_metrics['model']=='LGBM', 'threshold'].values[0]
Time_25 = forward_test_df.loc[(forward_test_df['Time']>=test_start_date) & (forward_test_df['Time']<=test_end_end), 'Time'].values
binary_predictions_2025_test = pd.DataFrame(
    {
        "cat_pred": (meta_X_test_25["cat_prob"] > test_2025_results_base_models['threshold'][0]).astype(int),
        "xgb_pred": (meta_X_test_25["xgb_prob"] > test_2025_results_base_models['threshold'][1]).astype(int),
        "logreg_pred": (meta_X_test_25["logreg_prob"] > test_2025_results_base_models['threshold'][2]).astype(int),
        "lgbm_pred": (meta_X_test_25["lgbm_prob"] > test_2025_results_base_models['threshold'][3]).astype(int),
        "Stack Ensemble": pd.Series(final_pred_25, name='Stack Ensemble'),
        "Time": Time_25,
        'True Class': y_test_25
    }
)
binary_predictions_2025_test.to_csv("binary_predictions_2025_test.csv", index=False)
print('Binary prediction file generated')

Index(['Feature_1', 'Feature_113', 'Feature_237', 'Feature_241',
       'Feature_1_x_Feature_113', 'Feature_1_x_Feature_237',
       'Feature_1_x_Feature_241', 'Feature_113_x_Feature_237',
       'Feature_113_x_Feature_241', 'Feature_237_x_Feature_241', 'Regime'],
      dtype='object')
Extreme filtering is done, original results 100 row, filtered 5
            model  threshold  precision  recall     f1  stability         confusion_matrix    score  score_1
0  Stack Ensemble      0.178      0.239   0.421  0.305       1.68  [[665, 296], [128, 93]] -10400.0  15200.0
Extreme filtering is done, original results 100 row, filtered 5
Extreme filtering is done, original results 100 row, filtered 56
Extreme filtering is done, original results 100 row, filtered 24
Extreme filtering is done, original results 100 row, filtered 16
      model  threshold  precision  recall     f1  stability         confusion_matrix    score  score_1
0  CatBoost      0.505      0.254   0.579  0.353      1.821  [[585, 3

In [16]:
# import pickle

# with open("./base_models/feature_scaler_lr.bin", "wb") as f:
#     pickle.dump(feature_scaler, f)

## Test Full Data

In [ ]:
###################################################################################################################
# Test results full
###################################################################################################################
full_data_df = train_val_df
full_data_df['Time'] = pd.to_datetime(full_data_df['Time'])
start_date = full_data_df['Time'].min()

end_date = full_data_df['Time'].max()

# Preprocess testing data
x_full, y_full = preprocess_test_data(test_df=full_data_df, start_date=start_date, end_date=end_date, scaler_gmm=scaler_gmm, gmm=gmm, intersected_features=intersected_features)

# Collect predictions from the fitted base models
cat_pred_full = cat_model.predict_proba(x_full)[:, 1]
xgb_pred_full = xgb_model.predict_proba(x_full)[:, 1]
lr_pred_full = lr_model.predict_proba(feature_scaler.transform(x_full))[:, 1]
lgbm_pred_full = lgb_model.predict_proba(x_full)[:, 1]

# Populate the predictions (inputs for testing on the meta learner)
meta_X_full = pd.DataFrame({"cat_prob": cat_pred_full,"xgb_prob": xgb_pred_full,"logreg_prob": lr_pred_full,"lgbm_prob": lgbm_pred_full})
meta_y_full = y_full

# Predict probability using the meta learner
meta_pred_prob_full = final_meta_learner.predict_proba(meta_X_full)[:, 1]

# Evaluate and print results of testing data
class1_proportion_full = y_full.value_counts(normalize=True)[1]
metrics_full = evaluate_model(y_full, meta_pred_prob_full, 'Stack Ensemble', class1_proportion_full)

# Saving evaluation metrics
optimal_thresh_full = metrics_full['threshold']

# Populate final stack ensemble predictions based on probabilities
final_pred_full = (meta_pred_prob_full > optimal_thresh_full).astype(int)

final_results_full = pd.DataFrame([metrics_full])

# Printing metrics
print(final_results_full.round(3).to_string(max_colwidth=50))
cat_metrics_full = evaluate_model(y_full, cat_pred_full, 'CatBoost', class1_proportion_full)
xgb_metrics_full = evaluate_model(y_full, xgb_pred_full, 'XGBoost', class1_proportion_full)
lr_metrics_full = evaluate_model(y_full, lr_pred_full, 'LogReg', class1_proportion_full)
lgbm_metrics_full = evaluate_model(y_full, lgbm_pred_full, 'LGBM', class1_proportion_full)

full_results_base_models = pd.DataFrame([cat_metrics_full,xgb_metrics_full,lr_metrics_full,lgbm_metrics_full])

print(full_results_base_models.round(3).to_string(max_colwidth=50))

CatBoost_thresh = full_data_metrics.loc[full_data_metrics['model']=='CatBoost', 'threshold'].values[0]
XGBoost_thresh = full_data_metrics.loc[full_data_metrics['model']=='XGBoost', 'threshold'].values[0]
LogReg_thresh = full_data_metrics.loc[full_data_metrics['model']=='LogReg', 'threshold'].values[0]
LGBM_thresh = full_data_metrics.loc[full_data_metrics['model']=='LGBM', 'threshold'].values[0]
Time_full = full_data_df.loc[(full_data_df['Time']>=start_date) & (full_data_df['Time']<=end_date), 'Time'].values

binary_predictions_full_test = pd.DataFrame(
    {
        "cat_pred": (meta_X_full["cat_prob"] > full_results_base_models['threshold'][0]).astype(int),
        "xgb_pred": (meta_X_full["xgb_prob"] > full_results_base_models['threshold'][1]).astype(int),
        "logreg_pred": (meta_X_full["logreg_prob"] > full_results_base_models['threshold'][2]).astype(int),
        "lgbm_pred": (meta_X_full["lgbm_prob"] > full_results_base_models['threshold'][3]).astype(int),
        "Stack Ensemble": pd.Series(final_pred_full, name='Stack Ensemble'),
        "Time": Time_full,
        'True Class': y_full
    }
)
binary_predictions_full_test.to_csv("binary_predictions_full.csv", index=False)
print('Binary prediction file generated')

Index(['Feature_1', 'Feature_113', 'Feature_237', 'Feature_241',
       'Feature_1_x_Feature_113', 'Feature_1_x_Feature_237',
       'Feature_1_x_Feature_241', 'Feature_113_x_Feature_237',
       'Feature_113_x_Feature_241', 'Feature_237_x_Feature_241', 'Regime'],
      dtype='object')
Extreme filtering is done, original results 100 row, filtered 10
            model  threshold  precision  recall     f1  stability             confusion_matrix     score   score_1
0  Stack Ensemble      0.149      0.208   0.985  0.344       1.26  [[395, 31961], [129, 8399]]  301200.0  327000.0
Extreme filtering is done, original results 100 row, filtered 8
Extreme filtering is done, original results 100 row, filtered 66
Extreme filtering is done, original results 100 row, filtered 48
Extreme filtering is done, original results 100 row, filtered 16
      model  threshold  precision  recall     f1  stability                confusion_matrix     score   score_1
0  CatBoost      0.436      0.209   0.939  0.34